<a href="https://colab.research.google.com/github/ibelalov/rlmw/blob/main/rlmw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# rlmw

Single-notebook development environment for low-weight span-vector search.

Core components:

1. Binary linear algebra over F_2
2. Planted span instance generator
3. Bitset/popcount utilities
4. Direction bank construction
5. Exact discrete-gradient descent
6. Local-minimum support-intersection constraints
7. Failed-local-minimum archive
8. Neural direction model
9. Cross-entropy sampler
10. Strategy-level Q-controller
11. Exact local-minimum intersection solver
12. Training/evaluation dashboard

In [17]:
# @title
# --- 00. Environment setup ---

import os
import sys
import math
import time
import json
import random
import pathlib
import itertools
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np

try:
    import torch
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("PyTorch available:", TORCH_AVAILABLE)
if TORCH_AVAILABLE:
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy: 2.0.2
PyTorch available: True
PyTorch: 2.10.0+cpu
CUDA available: False


In [18]:
# @title Environment detection and storage backend selection
import os


def _env_flag(name: str, default: str = "0") -> bool:
    return os.environ.get(name, default).strip().lower() in {"1", "true", "yes", "on"}


# Small self-tests for environment-flag parsing
assert _env_flag("__RLMW_TEST_UNSET__", "0") is False
assert _env_flag("__RLMW_TEST_UNSET__", "1") is True

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

HEADLESS = _env_flag("RLMW_HEADLESS", "0")
SMOKE = _env_flag("RLMW_SMOKE", "0")
USE_DRIVE = IN_COLAB and (not HEADLESS)

if USE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [19]:
# @title Project paths
import os
from pathlib import Path

if USE_DRIVE:
    PROJECT_ROOT = Path("/content/drive/MyDrive") / "rlmw"
else:
    PROJECT_ROOT = Path(os.environ.get("RLMW_PROJECT_ROOT", "/tmp/rlmw"))

DATA_DIR = PROJECT_ROOT / "data"
RUNS_DIR = PROJECT_ROOT / "runs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
EXPORT_DIR = PROJECT_ROOT / "exports"
CACHE_DIR = PROJECT_ROOT / "cache"

for p in [PROJECT_ROOT, DATA_DIR, RUNS_DIR, CHECKPOINT_DIR, EXPORT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("HEADLESS:", HEADLESS)
print("SMOKE:", SMOKE)
print("USE_DRIVE:", USE_DRIVE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("EXPORT_DIR:", EXPORT_DIR)
print("CACHE_DIR:", CACHE_DIR)

IN_COLAB: True
HEADLESS: False
SMOKE: False
USE_DRIVE: True
PROJECT_ROOT: /content/drive/MyDrive/rlmw
DATA_DIR: /content/drive/MyDrive/rlmw/data
RUNS_DIR: /content/drive/MyDrive/rlmw/runs
CHECKPOINT_DIR: /content/drive/MyDrive/rlmw/checkpoints
EXPORT_DIR: /content/drive/MyDrive/rlmw/exports
CACHE_DIR: /content/drive/MyDrive/rlmw/cache


In [ ]:
# @title Optional Colab dependency bootstrap
import subprocess


def _module_available(module_name: str) -> bool:
    try:
        __import__(module_name)
        return True
    except Exception:
        return False


if IN_COLAB and not HEADLESS:
    if not _module_available("ortools"):
        print("Installing missing Colab dependency: ortools")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ortools"])
        print("Installed ortools. Continue running the notebook from top to bottom.")
    else:
        print("Colab dependency available: ortools")
else:
    print("Dependency bootstrap skipped outside interactive Colab.")


## 01. Binary linear algebra over F_2

In [20]:
# @title GF(2) utilities (exact, uint8/binary)
import numpy as np


def as_binary_uint8(x, *, name="array", copy=True) -> np.ndarray:
    """Return `x` as a uint8 NumPy array containing only 0/1."""
    arr = np.array(x, copy=True) if copy else np.asarray(x)

    if arr.dtype == np.bool_:
        return arr.astype(np.uint8, copy=copy)

    if not np.issubdtype(arr.dtype, np.integer):
        raise TypeError(f"{name} must have boolean or integer dtype, got {arr.dtype!r}")

    mask = (arr == 0) | (arr == 1)
    if arr.size and not mask.all():
        bad = np.unique(arr[~mask])
        raise ValueError(f"{name} must be binary (0/1) only; found values {bad.tolist()}")

    return arr.astype(np.uint8, copy=copy)


def hamming_weight(x) -> int:
    arr = as_binary_uint8(x, name="x", copy=False)
    return int(arr.sum(dtype=np.int64))


def gf2_matvec(A, u) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    u_bin = as_binary_uint8(u, name="u", copy=False)

    if A_bin.ndim != 2:
        raise ValueError(f"A must be 2D, got shape {A_bin.shape}")
    if u_bin.ndim != 1:
        raise ValueError(f"u must be 1D, got shape {u_bin.shape}")
    if A_bin.shape[1] != u_bin.shape[0]:
        raise ValueError(f"shape mismatch: A is {A_bin.shape}, u is {u_bin.shape}")

    y = (A_bin.astype(np.int64) @ u_bin.astype(np.int64)) & 1
    return y.astype(np.uint8)


def gf2_matmul(A, B) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    B_bin = as_binary_uint8(B, name="B", copy=False)

    if A_bin.ndim != 2 or B_bin.ndim != 2:
        raise ValueError(f"A and B must be 2D, got {A_bin.shape} and {B_bin.shape}")
    if A_bin.shape[1] != B_bin.shape[0]:
        raise ValueError(f"shape mismatch: A is {A_bin.shape}, B is {B_bin.shape}")

    C = (A_bin.astype(np.int64) @ B_bin.astype(np.int64)) & 1
    return C.astype(np.uint8)


def gf2_rank(A) -> int:
    M = as_binary_uint8(A, name="A", copy=True)
    if M.ndim != 2:
        raise ValueError(f"A must be 2D, got shape {M.shape}")

    rows, cols = M.shape
    r = 0
    for c in range(cols):
        pivot = None
        for i in range(r, rows):
            if M[i, c] == 1:
                pivot = i
                break
        if pivot is None:
            continue

        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]

        for i in range(r + 1, rows):
            if M[i, c] == 1:
                M[i, :] ^= M[r, :]

        r += 1
        if r == rows:
            break

    return int(r)


def gf2_random_matrix(rows, cols, rng=None, density=0.5) -> np.ndarray:
    if rows < 0 or cols < 0:
        raise ValueError("rows and cols must be nonnegative")
    if not (0.0 <= density <= 1.0):
        raise ValueError("density must be in [0, 1]")

    rng = np.random.default_rng() if rng is None else rng
    A = (rng.random((rows, cols)) < density).astype(np.uint8)
    return as_binary_uint8(A, name="A", copy=False)


def gf2_random_invertible(n, rng=None, max_tries=1000) -> np.ndarray:
    if n < 0:
        raise ValueError("n must be nonnegative")
    if max_tries <= 0:
        raise ValueError("max_tries must be positive")

    rng = np.random.default_rng() if rng is None else rng

    for _ in range(max_tries):
        Q = gf2_random_matrix(n, n, rng=rng, density=0.5)
        if gf2_rank(Q) == n:
            return Q

    raise RuntimeError(f"failed to sample invertible {n}x{n} matrix within {max_tries} tries")


def gf2_inverse(A) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=True)
    if A_bin.ndim != 2:
        raise ValueError(f"A must be 2D, got shape {A_bin.shape}")

    n, m = A_bin.shape
    if n != m:
        raise ValueError(f"A must be square, got shape {A_bin.shape}")

    I = np.eye(n, dtype=np.uint8)
    aug = np.concatenate([A_bin, I], axis=1)

    for c in range(n):
        pivot = None
        for i in range(c, n):
            if aug[i, c] == 1:
                pivot = i
                break
        if pivot is None:
            raise ValueError("A is singular over F_2")

        if pivot != c:
            aug[[c, pivot]] = aug[[pivot, c]]

        for i in range(n):
            if i != c and aug[i, c] == 1:
                aug[i, :] ^= aug[c, :]

    inv = aug[:, n:]
    return as_binary_uint8(inv, name="A_inv", copy=False)


def support_intersection(c, d) -> int:
    c_bin = as_binary_uint8(c, name="c", copy=False)
    d_bin = as_binary_uint8(d, name="d", copy=False)

    if c_bin.ndim != 1 or d_bin.ndim != 1:
        raise ValueError(f"c and d must be 1D, got {c_bin.shape} and {d_bin.shape}")
    if c_bin.shape != d_bin.shape:
        raise ValueError(f"shape mismatch: c is {c_bin.shape}, d is {d_bin.shape}")

    return int(np.bitwise_and(c_bin, d_bin).sum(dtype=np.int64))


def directional_delta(c, d) -> int:
    c_bin = as_binary_uint8(c, name="c", copy=False)
    d_bin = as_binary_uint8(d, name="d", copy=False)

    if c_bin.ndim != 1 or d_bin.ndim != 1:
        raise ValueError(f"c and d must be 1D, got {c_bin.shape} and {d_bin.shape}")
    if c_bin.shape != d_bin.shape:
        raise ValueError(f"shape mismatch: c is {c_bin.shape}, d is {d_bin.shape}")

    return hamming_weight(d_bin) - 2 * support_intersection(c_bin, d_bin)


def run_f2_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    # Binary/uint8 conversion checks
    v = as_binary_uint8([0, 1, 1, 0], name="v")
    assert v.dtype == np.uint8 and np.isin(v, [0, 1]).all()
    b = as_binary_uint8(np.array([True, False, True]), name="b")
    assert b.dtype == np.uint8 and np.array_equal(b, np.array([1, 0, 1], dtype=np.uint8))

    try:
        as_binary_uint8(np.array([0, 2], dtype=np.int64), name="bad")
        raise AssertionError("expected ValueError for non-binary integers")
    except ValueError:
        pass

    try:
        as_binary_uint8(np.array([0.0, 1.0]), name="float_bad")
        raise AssertionError("expected TypeError for non-integer dtype")
    except TypeError:
        pass


    for overflow_bad in [
        np.array([256], dtype=np.int64),
        np.array([-256], dtype=np.int64),
        np.array([0, 1, 257], dtype=np.int64),
        np.array([256], dtype=np.uint64),
    ]:
        try:
            as_binary_uint8(overflow_bad, name="overflow_bad")
            raise AssertionError("expected ValueError for overflow-prone non-binary integers")
        except ValueError:
            pass

    # Edge vectors
    z = as_binary_uint8(np.zeros(6, dtype=np.uint8), name="z")
    oh = as_binary_uint8(np.array([0, 0, 1, 0, 0, 0], dtype=np.uint8), name="oh")
    ones = as_binary_uint8(np.ones(6, dtype=np.uint8), name="ones")
    assert hamming_weight([0, 1, 1, 0]) == 2
    y_list = gf2_matvec([[1, 0], [1, 1]], [1, 1])
    assert y_list.dtype == np.uint8 and np.array_equal(y_list, np.array([1, 0], dtype=np.uint8))

    assert hamming_weight(z) == 0
    assert hamming_weight(oh) == 1
    assert hamming_weight(ones) == 6

    # Matvec: compare with direct parity loops
    for _ in range(20):
        n = int(rng.integers(1, 7))
        r = int(rng.integers(1, 7))
        A = gf2_random_matrix(n, r, rng=rng, density=0.5)
        u = gf2_random_matrix(r, 1, rng=rng, density=0.5).reshape(-1)
        y = gf2_matvec(A, u)

        y_ref = np.zeros(n, dtype=np.uint8)
        for i in range(n):
            s = 0
            for j in range(r):
                s ^= int(A[i, j] & u[j])
            y_ref[i] = s

        assert y.dtype == np.uint8 and y.shape == (n,)
        assert np.array_equal(y, y_ref)

    # Matmul: compare with direct parity loops
    for _ in range(20):
        n = int(rng.integers(1, 6))
        r = int(rng.integers(1, 6))
        k = int(rng.integers(1, 6))
        A = gf2_random_matrix(n, r, rng=rng, density=0.5)
        B = gf2_random_matrix(r, k, rng=rng, density=0.5)
        C = gf2_matmul(A, B)

        C_ref = np.zeros((n, k), dtype=np.uint8)
        for i in range(n):
            for j in range(k):
                s = 0
                for t in range(r):
                    s ^= int(A[i, t] & B[t, j])
                C_ref[i, j] = s

        assert C.dtype == np.uint8 and C.shape == (n, k)
        assert np.array_equal(C, C_ref)

    # Rank edge cases
    Z = np.zeros((4, 5), dtype=np.uint8)
    assert gf2_rank(Z) == 0

    I5 = np.eye(5, dtype=np.uint8)
    assert gf2_rank(I5) == 5

    D = np.array([
        [1, 0, 1, 1],
        [1, 0, 1, 1],
        [0, 1, 1, 0],
    ], dtype=np.uint8)
    assert gf2_rank(D) == 2

    for _ in range(20):
        n = int(rng.integers(1, 7))
        m = int(rng.integers(1, 7))
        A = gf2_random_matrix(n, m, rng=rng, density=0.5)
        rnk = gf2_rank(A)
        assert isinstance(rnk, int)
        assert 0 <= rnk <= min(n, m)

    # Small rectangular matrices
    A_rect = gf2_random_matrix(2, 5, rng=rng)
    B_rect = gf2_random_matrix(5, 3, rng=rng)
    C_rect = gf2_matmul(A_rect, B_rect)
    assert C_rect.shape == (2, 3)
    assert C_rect.dtype == np.uint8 and np.isin(C_rect, [0, 1]).all()

    # Invertible sampling and inverse checks
    for n in [1, 2, 3, 5]:
        Q = gf2_random_invertible(n, rng=rng, max_tries=2000)
        Q_inv = gf2_inverse(Q)
        I = np.eye(n, dtype=np.uint8)
        assert np.array_equal(gf2_matmul(Q, Q_inv), I)
        assert np.array_equal(gf2_matmul(Q_inv, Q), I)

    # singular / non-square inverse errors
    try:
        gf2_inverse(np.array([[1, 0, 1]], dtype=np.uint8))
        raise AssertionError("expected ValueError for non-square matrix")
    except ValueError:
        pass

    try:
        gf2_inverse(np.array([[1, 1], [1, 1]], dtype=np.uint8))
        raise AssertionError("expected ValueError for singular matrix")
    except ValueError:
        pass

    # Directional delta identities
    for _ in range(40):
        n = int(rng.integers(1, 20))
        c = gf2_random_matrix(n, 1, rng=rng).reshape(-1)
        d = gf2_random_matrix(n, 1, rng=rng).reshape(-1)

        delta = directional_delta(c, d)
        lhs1 = hamming_weight(c ^ d) - hamming_weight(c)
        lhs2 = hamming_weight(d) - 2 * support_intersection(c, d)

        assert delta == lhs1
        assert delta == lhs2

    # explicit edge cases for directional_delta
    assert directional_delta(z, z) == 0
    assert directional_delta(z, oh) == 1
    assert directional_delta(ones, ones) == -6

    print("run_f2_self_tests: all tests passed")

In [21]:
run_f2_self_tests()

run_f2_self_tests: all tests passed


## 02. Planted span-instance generator

In [22]:
# @title Planted span-instance generator (exact, uint8/binary)
from dataclasses import dataclass


@dataclass
class SpanInstance:
    A: np.ndarray
    c_star: np.ndarray
    u_star: np.ndarray
    W: int
    metadata: dict

    def matvec(self, u) -> np.ndarray:
        return gf2_matvec(self.A, u)

    def weight(self, c) -> int:
        return hamming_weight(c)

    def verify(self) -> None:
        A = as_binary_uint8(self.A, name="A", copy=False)
        c_star = as_binary_uint8(self.c_star, name="c_star", copy=False)
        u_star = as_binary_uint8(self.u_star, name="u_star", copy=False)

        if A.ndim != 2:
            raise ValueError(f"A must be 2D, got shape {A.shape}")
        N, r = A.shape

        if c_star.ndim != 1 or c_star.shape != (N,):
            raise ValueError(f"c_star must have shape ({N},), got {c_star.shape}")
        if u_star.ndim != 1 or u_star.shape != (r,):
            raise ValueError(f"u_star must have shape ({r},), got {u_star.shape}")

        if hamming_weight(u_star) == 0:
            raise ValueError("u_star must be nonzero")
        if int(self.W) != hamming_weight(c_star):
            raise ValueError("W must equal hamming_weight(c_star)")

        if not np.array_equal(gf2_matvec(A, u_star), c_star):
            raise ValueError("constraint violated: A u_star != c_star")


def generate_planted_span_instance(
    N: int,
    r: int,
    W: int,
    seed=None,
    density: float = 0.5,
    hide: bool = True,
) -> SpanInstance:
    if N <= 0:
        raise ValueError("N must be positive")
    if r <= 0:
        raise ValueError("r must be positive")
    if not (1 <= W <= N):
        raise ValueError("W must satisfy 1 <= W <= N")
    if not (0.0 <= density <= 1.0):
        raise ValueError("density must be in [0, 1]")

    rng = np.random.default_rng(seed)

    support = rng.choice(N, size=W, replace=False)
    c_star = np.zeros(N, dtype=np.uint8)
    c_star[support] = 1

    u_star = np.zeros(r, dtype=np.uint8)
    while hamming_weight(u_star) == 0:
        u_star = (rng.random(r) < 0.5).astype(np.uint8)

    one_positions = np.flatnonzero(u_star)
    k = int(rng.choice(one_positions))

    A = gf2_random_matrix(N, r, rng=rng, density=density)

    parity = np.zeros(N, dtype=np.uint8)
    for j in one_positions:
        if j != k:
            parity ^= A[:, j]
    A[:, k] = c_star ^ parity

    if hide:
        Q = gf2_random_invertible(r, rng=rng)
        Q_inv = gf2_inverse(Q)
        A = gf2_matmul(A, Q)
        u_star = gf2_matvec(Q_inv, u_star)

    if not np.array_equal(gf2_matvec(A, u_star), c_star):
        raise RuntimeError("internal construction error: A u_star != c_star")

    metadata = {
        "N": int(N),
        "r": int(r),
        "W": int(W),
        "seed": seed,
        "density": float(density),
        "hide": bool(hide),
    }
    inst = SpanInstance(A=A, c_star=c_star, u_star=u_star, W=int(W), metadata=metadata)
    inst.verify()
    return inst


def run_planted_instance_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    for hide in [False, True]:
        for _ in range(8):
            N = int(rng.integers(4, 13))
            r = int(rng.integers(2, 11))
            W = int(rng.integers(1, N + 1))
            density = float(rng.uniform(0.0, 1.0))
            s = int(rng.integers(0, 10**9))

            inst = generate_planted_span_instance(N=N, r=r, W=W, seed=s, density=density, hide=hide)
            inst.verify()

            assert inst.weight(inst.c_star) == W
            assert np.array_equal(inst.matvec(inst.u_star), inst.c_star)

            for arr in [inst.A, inst.c_star, inst.u_star]:
                assert arr.dtype == np.uint8
                assert np.isin(arr, [0, 1]).all()

    params = dict(N=10, r=6, W=3, seed=12345, density=0.4, hide=True)
    a = generate_planted_span_instance(**params)
    b = generate_planted_span_instance(**params)
    assert np.array_equal(a.A, b.A)
    assert np.array_equal(a.c_star, b.c_star)
    assert np.array_equal(a.u_star, b.u_star)

    invalid_cases = [
        dict(N=10, r=5, W=0, seed=0, density=0.5, hide=False),
        dict(N=10, r=5, W=11, seed=0, density=0.5, hide=False),
        dict(N=0, r=5, W=1, seed=0, density=0.5, hide=False),
        dict(N=10, r=0, W=1, seed=0, density=0.5, hide=False),
        dict(N=10, r=5, W=1, seed=0, density=-0.1, hide=False),
        dict(N=10, r=5, W=1, seed=0, density=1.1, hide=False),
    ]
    for kwargs in invalid_cases:
        try:
            generate_planted_span_instance(**kwargs)
            raise AssertionError(f"expected ValueError for invalid kwargs={kwargs}")
        except ValueError:
            pass

    print("run_planted_instance_self_tests: all tests passed")

In [23]:
run_planted_instance_self_tests()

run_planted_instance_self_tests: all tests passed


## 03. Direction bank

In [24]:
# @title Direction bank (exact span directions)

class DirectionBank:
    def __init__(self, A):
        A_bin = as_binary_uint8(A, name="A", copy=True)
        if A_bin.ndim != 2:
            raise ValueError(f"A must have shape [N, r], got ndim={A_bin.ndim}")

        N, r = A_bin.shape
        self.A = A_bin
        self.V = np.zeros((0, r), dtype=np.uint8)
        self.D = np.zeros((0, N), dtype=np.uint8)
        self.weights = np.zeros((0,), dtype=np.int64)
        self.hashes = set()
        self.usage_counts = np.zeros((0,), dtype=np.int64)
        self.ages = np.zeros((0,), dtype=np.int64)

    def _direction_key(self, d: np.ndarray) -> bytes:
        return np.ascontiguousarray(d, dtype=np.uint8).tobytes()

    def add_one(self, v):
        v_bin = as_binary_uint8(v, name="v", copy=False)
        if v_bin.ndim != 1 or v_bin.shape[0] != self.A.shape[1]:
            raise ValueError(f"v must have shape [{self.A.shape[1]}], got {v_bin.shape}")

        d = gf2_matvec(self.A, v_bin)
        if hamming_weight(d) == 0:
            return None

        key = self._direction_key(d)
        if key in self.hashes:
            return None

        idx = self.D.shape[0]
        self.V = np.vstack([self.V, v_bin.reshape(1, -1)])
        self.D = np.vstack([self.D, d.reshape(1, -1)])
        self.weights = np.append(self.weights, hamming_weight(d))
        self.usage_counts = np.append(self.usage_counts, 0)
        self.ages = np.append(self.ages, 0)
        self.hashes.add(key)
        return idx

    def add_coefficients(self, V_new):
        Vb = as_binary_uint8(V_new, name="V_new", copy=False)
        if Vb.ndim == 1:
            rows = [Vb]
        elif Vb.ndim == 2 and Vb.shape[1] == self.A.shape[1]:
            rows = [Vb[i] for i in range(Vb.shape[0])]
        else:
            raise ValueError(f"V_new must have shape [{self.A.shape[1]}] or [K, {self.A.shape[1]}], got {Vb.shape}")

        accepted = []
        for row in rows:
            idx = self.add_one(row)
            if idx is not None:
                accepted.append(idx)
        return accepted

    def add_random_span_directions(self, K, rng, max_attempts=None):
        if K < 0:
            raise ValueError("K must be nonnegative")
        if max_attempts is None:
            max_attempts = max(100, 20 * K)

        accepted = 0
        attempts = 0
        r = self.A.shape[1]
        while accepted < K and attempts < max_attempts:
            v = rng.integers(0, 2, size=(r,), dtype=np.uint8)
            idx = self.add_one(v)
            if idx is not None:
                accepted += 1
            attempts += 1
        return accepted

    def deltas(self, c) -> np.ndarray:
        c_bin = as_binary_uint8(c, name="c", copy=False)
        if c_bin.ndim != 1 or c_bin.shape[0] != self.A.shape[0]:
            raise ValueError(f"c must have shape [{self.A.shape[0]}], got {c_bin.shape}")

        M = self.D.shape[0]
        if M == 0:
            return np.zeros((0,), dtype=np.int64)

        intersections = np.sum(self.D & c_bin.reshape(1, -1), axis=1, dtype=np.int64)
        return self.weights.astype(np.int64) - 2 * intersections

    def best_descent(self, c) -> Optional[int]:
        c_bin = as_binary_uint8(c, name="c", copy=False)
        deltas = self.deltas(c_bin)
        if deltas.size == 0:
            return None

        candidates = np.where(deltas < 0)[0]
        if candidates.size == 0:
            return None

        for idx in candidates[np.argsort(deltas[candidates])]:
            c_new = c_bin ^ self.D[idx]
            if hamming_weight(c_new) != 0:
                return int(idx)
        return None

    def features_for_ranker(self, c) -> np.ndarray:
        c_bin = as_binary_uint8(c, name="c", copy=False)
        M = self.D.shape[0]
        if M == 0:
            return np.zeros((0, 6), dtype=np.float32)

        N = float(self.A.shape[0])
        intersections = np.sum(self.D & c_bin.reshape(1, -1), axis=1, dtype=np.int64)
        deltas = self.weights.astype(np.int64) - 2 * intersections

        age_denom = max(1.0, float(np.max(self.ages)) if self.ages.size else 1.0)
        use_denom = max(1.0, float(np.max(self.usage_counts)) if self.usage_counts.size else 1.0)

        feats = np.stack([
            self.weights / N,
            intersections / N,
            deltas / N,
            (deltas < 0).astype(np.float32),
            self.ages / age_denom,
            self.usage_counts / use_denom,
        ], axis=1)
        return feats.astype(np.float32)

    def verify(self) -> None:
        A = as_binary_uint8(self.A, name="A", copy=False)
        if A.ndim != 2:
            raise ValueError("A must be 2D")
        N, r = A.shape
        M = self.D.shape[0]

        if self.V.shape != (M, r):
            raise ValueError(f"V shape mismatch: expected {(M, r)}, got {self.V.shape}")
        if self.D.shape != (M, N):
            raise ValueError(f"D shape mismatch: expected {(M, N)}, got {self.D.shape}")

        _ = as_binary_uint8(self.V, name="V", copy=False)
        _ = as_binary_uint8(self.D, name="D", copy=False)

        if len(self.weights) != M or len(self.usage_counts) != M or len(self.ages) != M:
            raise ValueError("weights/usage_counts/ages length mismatch")

        if len(self.hashes) != M:
            raise ValueError("hash set size mismatch")

        rebuilt_hashes = set()
        for i in range(M):
            d = gf2_matvec(A, self.V[i])
            if not np.array_equal(d, self.D[i]):
                raise ValueError(f"direction mismatch at index {i}")
            w = hamming_weight(self.D[i])
            if w == 0:
                raise ValueError(f"zero direction at index {i}")
            if int(self.weights[i]) != w:
                raise ValueError(f"weight mismatch at index {i}")
            key = self._direction_key(self.D[i])
            if key in rebuilt_hashes:
                raise ValueError("duplicate direction hash")
            rebuilt_hashes.add(key)

        if rebuilt_hashes != self.hashes:
            raise ValueError("stored hashes do not match directions")


def run_direction_bank_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)
    randvec = lambda n: rng.integers(0, 2, size=(n,), dtype=np.uint8)
    A = gf2_random_matrix(12, 8, rng=rng)
    bank = DirectionBank(A)

    accepted = bank.add_random_span_directions(20, rng=rng)
    assert accepted >= 1
    bank.verify()

    for i in range(bank.D.shape[0]):
        assert np.array_equal(bank.D[i], gf2_matvec(bank.A, bank.V[i]))
        assert hamming_weight(bank.D[i]) > 0

    v = bank.V[0].copy()
    M_before = bank.D.shape[0]
    assert bank.add_one(v) is None
    assert bank.D.shape[0] == M_before

    z_found = None
    for _ in range(256):
        z = randvec(A.shape[1])
        if hamming_weight(z) > 0 and hamming_weight(gf2_matvec(A, z)) == 0:
            z_found = z
            break
    if z_found is not None:
        assert bank.add_one(v ^ z_found) is None

    empty = DirectionBank(A)
    c0 = randvec(A.shape[0])
    assert empty.deltas(c0).shape == (0,)
    assert empty.best_descent(c0) is None

    c = randvec(A.shape[0])
    deltas = bank.deltas(c)
    for i in range(bank.D.shape[0]):
        assert int(deltas[i]) == directional_delta(c, bank.D[i])

    feats = bank.features_for_ranker(c)
    assert feats.shape[0] == bank.D.shape[0]
    assert feats.shape[1] >= 4
    assert np.isfinite(feats).all()


run_direction_bank_self_tests()

## 04. Exact discrete-gradient descent

In [25]:
# @title Exact discrete-gradient descent

class GradientEngine:
    def delta(self, c, d) -> int:
        c_bin = as_binary_uint8(c, name="c", copy=False)
        d_bin = as_binary_uint8(d, name="d", copy=False)
        if c_bin.shape != d_bin.shape:
            raise ValueError(f"c and d must share shape, got {c_bin.shape} vs {d_bin.shape}")

        delta_1 = directional_delta(c_bin, d_bin)
        delta_2 = hamming_weight(c_bin ^ d_bin) - hamming_weight(c_bin)
        if int(delta_1) != int(delta_2):
            raise ValueError("delta formula mismatch")
        return int(delta_1)

    def apply_best(self, c, bank) -> tuple[np.ndarray, dict]:
        c_bin = as_binary_uint8(c, name="c", copy=True)
        if c_bin.ndim != 1 or c_bin.shape[0] != bank.A.shape[0]:
            raise ValueError(f"c must have shape [{bank.A.shape[0]}], got {c_bin.shape}")

        idx = bank.best_descent(c_bin)
        if idx is None:
            return c_bin.copy(), {"moved": False, "reason": "no_descent"}

        d = bank.D[idx]
        delta = self.delta(c_bin, d)
        old_weight = hamming_weight(c_bin)
        c_new = c_bin ^ d
        new_weight = hamming_weight(c_new)

        if new_weight == 0:
            raise ValueError("invalid move: moved to zero")
        if new_weight >= old_weight or delta >= 0:
            raise ValueError("invalid move: weight did not strictly decrease")

        intersection = int(np.sum(c_bin & d))
        bank.usage_counts[idx] += 1

        meta = {
            "moved": True,
            "idx": int(idx),
            "delta": int(delta),
            "old_weight": int(old_weight),
            "new_weight": int(new_weight),
            "direction_weight": int(bank.weights[idx]),
            "intersection": intersection,
        }
        return c_new, meta

    def descend(self, c, bank, max_steps) -> tuple[np.ndarray, list[dict]]:
        if max_steps < 0:
            raise ValueError("max_steps must be nonnegative")
        current = as_binary_uint8(c, name="c", copy=True)
        history = []

        for _ in range(max_steps):
            nxt, meta = self.apply_best(current, bank)
            history.append(meta)
            if not meta.get("moved", False):
                break
            if hamming_weight(nxt) >= hamming_weight(current):
                raise ValueError("descend step failed to strictly decrease")
            bank.ages += 1
            bank.ages[meta["idx"]] = 0
            current = nxt
        return current, history


def run_gradient_engine_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)
    randvec = lambda n: rng.integers(0, 2, size=(n,), dtype=np.uint8)
    engine = GradientEngine()

    c = randvec(20)
    d = randvec(20)
    delta = engine.delta(c, d)
    assert delta == hamming_weight(c ^ d) - hamming_weight(c)
    assert delta == hamming_weight(d) - 2 * int(np.sum(c & d))

    A = np.eye(4, dtype=np.uint8)
    bank = DirectionBank(A)
    c_zero_block = np.array([1, 1, 0, 0], dtype=np.uint8)
    bank.add_one(c_zero_block)
    c2, m2 = engine.apply_best(c_zero_block, bank)
    assert np.array_equal(c2, c_zero_block)
    assert m2["moved"] is False and m2["reason"] == "no_descent"

    A3 = gf2_random_matrix(16, 10, rng=rng)
    bank3 = DirectionBank(A3)
    bank3.add_random_span_directions(40, rng=rng)
    u = randvec(A3.shape[1])
    c3 = gf2_matvec(A3, u)
    if hamming_weight(c3) == 0:
        for _ in range(256):
            u = randvec(A3.shape[1])
            c3 = gf2_matvec(A3, u)
            if hamming_weight(c3) > 0:
                break


    c_next, meta = engine.apply_best(c3, bank3)
    if meta["moved"]:
        assert hamming_weight(c_next) < hamming_weight(c3)

    u0 = randvec(A3.shape[1])
    v0 = randvec(A3.shape[1])
    left = gf2_matvec(A3, u0) ^ gf2_matvec(A3, v0)
    right = gf2_matvec(A3, u0 ^ v0)
    assert np.array_equal(left, right)

    final_c, hist = engine.descend(c3, bank3, max_steps=25)
    prev_w = hamming_weight(c3)
    for h in hist:
        if h.get("moved", False):
            assert h["new_weight"] < h["old_weight"]
            assert h["old_weight"] <= prev_w
            prev_w = h["new_weight"]
    assert hamming_weight(final_c) <= hamming_weight(c3)


run_gradient_engine_self_tests()

## 05. Local-minimum intersection solver

In [26]:
# @title Local-minimum intersection solver (exact CP-SAT feasibility)
try:
    from ortools.sat.python import cp_model
    ORTOOLS_AVAILABLE = True
except Exception as e:
    cp_model = None
    ORTOOLS_AVAILABLE = False
    ORTOOLS_IMPORT_ERROR = e


@dataclass
class SolverResult:
    status: str
    found: bool
    u: Optional[np.ndarray]
    c: Optional[np.ndarray]
    weight: Optional[int]
    runtime_s: float
    num_variables: int
    num_constraints: int
    num_directions: int
    metadata: dict


class LocalMinimumSolver:
    def __init__(
        self,
        num_search_workers: int = 1,
        random_seed: int = 0,
        log_search_progress: bool = False,
    ):
        self.num_search_workers = int(num_search_workers)
        self.random_seed = int(random_seed)
        self.log_search_progress = bool(log_search_progress)

    def solve(self, A, W, direction_bank, time_limit_s: float = 10.0) -> SolverResult:
        if not ORTOOLS_AVAILABLE:
            raise RuntimeError(
                f"OR-Tools unavailable; cannot run LocalMinimumSolver.solve. Import error: {ORTOOLS_IMPORT_ERROR!r}"
            )

        A_bin = as_binary_uint8(A, name='A', copy=False)
        if A_bin.ndim != 2:
            raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
        N, r = A_bin.shape
        if N <= 0 or r <= 0:
            raise ValueError(f"A must have positive shape [N, r], got {A_bin.shape}")

        if int(W) != W:
            raise ValueError('W must be an integer')
        W = int(W)
        if not (1 <= W <= N):
            raise ValueError(f'W must satisfy 1 <= W <= N={N}, got {W}')

        direction_bank.verify()
        if direction_bank.A.shape != A_bin.shape or not np.array_equal(direction_bank.A, A_bin):
            raise ValueError('direction_bank.A must match A exactly')

        D = direction_bank.D
        weights = direction_bank.weights.astype(np.int64, copy=False)
        num_directions = int(D.shape[0])

        model = cp_model.CpModel()
        num_constraints = 0

        u_vars = [model.NewBoolVar(f'u_{j}') for j in range(r)]
        c_vars = [model.NewBoolVar(f'c_{i}') for i in range(N)]

        for i in range(N):
            S_i = np.flatnonzero(A_bin[i])
            if S_i.size == 0:
                model.Add(c_vars[i] == 0)
                num_constraints += 1
            else:
                model.AddBoolXOr([u_vars[int(j)] for j in S_i] + [c_vars[i].Not()])
                num_constraints += 1

        model.Add(sum(c_vars) >= 1)
        model.Add(sum(c_vars) <= W)
        num_constraints += 2

        for j in range(num_directions):
            supp = np.flatnonzero(D[j])
            rhs = int(weights[j] // 2)
            model.Add(sum(c_vars[int(i)] for i in supp) <= rhs)
            num_constraints += 1

        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = float(time_limit_s)
        solver.parameters.num_search_workers = self.num_search_workers
        solver.parameters.random_seed = self.random_seed
        solver.parameters.log_search_progress = self.log_search_progress

        status_code = solver.Solve(model)
        status_name = solver.StatusName(status_code)
        runtime_s = float(solver.WallTime())

        if status_code in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            u = np.array([solver.Value(v) for v in u_vars], dtype=np.uint8)
            c = np.array([solver.Value(v) for v in c_vars], dtype=np.uint8)

            if not np.array_equal(c, gf2_matvec(A_bin, u)):
                raise ValueError('solver returned inconsistent assignment: c != A u over F_2')
            if hamming_weight(c) == 0:
                raise ValueError('solver returned zero codeword despite wt(c) >= 1 constraint')
            if hamming_weight(c) > W:
                raise ValueError('solver returned wt(c) > W')

            for j in range(num_directions):
                lhs = int(np.sum(c & D[j]))
                rhs = int(weights[j] // 2)
                if lhs > rhs:
                    raise ValueError(f'local-minimum inequality violated at direction {j}: {lhs} > {rhs}')

            return SolverResult(
                status=status_name,
                found=True,
                u=u,
                c=c,
                weight=hamming_weight(c),
                runtime_s=runtime_s,
                num_variables=int(r + N),
                num_constraints=num_constraints,
                num_directions=num_directions,
                metadata={'status_code': int(status_code), 'status_name': status_name},
            )

        return SolverResult(
            status=status_name,
            found=False,
            u=None,
            c=None,
            weight=None,
            runtime_s=runtime_s,
            num_variables=int(r + N),
            num_constraints=num_constraints,
            num_directions=num_directions,
            metadata={'status_code': int(status_code), 'status_name': status_name},
        )


def verify_solver_result(A, W, direction_bank, result) -> None:
    if not result.found:
        return

    A_bin = as_binary_uint8(A, name='A', copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f'A must have shape [N, r], got {A_bin.shape}')
    N, r = A_bin.shape

    c = as_binary_uint8(result.c, name='result.c', copy=False)
    u = as_binary_uint8(result.u, name='result.u', copy=False)

    if c.ndim != 1 or c.shape != (N,):
        raise ValueError(f'result.c must have shape ({N},), got {c.shape}')
    if u.ndim != 1 or u.shape != (r,):
        raise ValueError(f'result.u must have shape ({r},), got {u.shape}')

    if not np.array_equal(c, gf2_matvec(A_bin, u)):
        raise ValueError('verification failed: c != A u over F_2')
    if hamming_weight(c) == 0:
        raise ValueError('verification failed: c must be nonzero')
    if hamming_weight(c) > int(W):
        raise ValueError('verification failed: wt(c) > W')

    direction_bank.verify()
    if direction_bank.A.shape != A_bin.shape or not np.array_equal(direction_bank.A, A_bin):
        raise ValueError('verification failed: direction_bank.A must match A exactly')

    for j in range(direction_bank.D.shape[0]):
        lhs = int(np.sum(c & direction_bank.D[j]))
        rhs = int(direction_bank.weights[j] // 2)
        if lhs > rhs:
            raise ValueError(f'verification failed at direction {j}: {lhs} > {rhs}')


def run_local_minimum_solver_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    inst = generate_planted_span_instance(N=20, r=8, W=3, seed=int(rng.integers(0, 10**9)), hide=True)
    empty_bank = DirectionBank(inst.A)
    solver = LocalMinimumSolver(num_search_workers=1, random_seed=seed)
    result = solver.solve(inst.A, W=inst.W, direction_bank=empty_bank, time_limit_s=10.0)
    assert result.found is True
    verify_solver_result(inst.A, inst.W, empty_bank, result)
    assert result.weight <= inst.W

    A4 = np.eye(4, dtype=np.uint8)
    W4 = 1
    bank4 = DirectionBank(A4)
    bank4.add_one(np.array([1, 0, 0, 0], dtype=np.uint8))
    bank4.add_one(np.array([0, 1, 0, 0], dtype=np.uint8))
    bank4.add_one(np.array([0, 0, 1, 0], dtype=np.uint8))
    result4 = solver.solve(A4, W=W4, direction_bank=bank4, time_limit_s=10.0)
    assert result4.found is True
    verify_solver_result(A4, W4, bank4, result4)
    assert np.array_equal(result4.c, np.array([0, 0, 0, 1], dtype=np.uint8))

    A3 = np.eye(3, dtype=np.uint8)
    W3 = 1
    bank3 = DirectionBank(A3)
    bank3.add_one(np.array([1, 0, 0], dtype=np.uint8))
    bank3.add_one(np.array([0, 1, 0], dtype=np.uint8))
    bank3.add_one(np.array([0, 0, 1], dtype=np.uint8))
    result3 = solver.solve(A3, W=W3, direction_bank=bank3, time_limit_s=10.0)
    assert result3.found is False
    assert result3.status in ('INFEASIBLE', 'MODEL_INVALID', 'UNKNOWN')

    sanity = result4
    assert np.array_equal(gf2_matvec(A4, sanity.u), sanity.c)
    assert 1 <= hamming_weight(sanity.c) <= W4
    for j in range(bank4.D.shape[0]):
        assert int(np.sum(sanity.c & bank4.D[j])) <= int(bank4.weights[j] // 2)

    try:
        solver.solve(A4, W=0, direction_bank=bank4, time_limit_s=1.0)
        raise AssertionError('expected ValueError for W=0')
    except ValueError:
        pass

    try:
        solver.solve(A4, W=5, direction_bank=bank4, time_limit_s=1.0)
        raise AssertionError('expected ValueError for W>N')
    except ValueError:
        pass

    try:
        solver.solve(np.array([1, 0, 1], dtype=np.uint8), W=1, direction_bank=bank3, time_limit_s=1.0)
        raise AssertionError('expected ValueError for non-2D A')
    except ValueError:
        pass

    bad_bank = DirectionBank(np.eye(5, dtype=np.uint8))
    try:
        solver.solve(A4, W=1, direction_bank=bad_bank, time_limit_s=1.0)
        raise AssertionError('expected ValueError for direction-bank mismatch')
    except ValueError:
        pass

    print('LocalMinimumSolver self-tests passed.')


In [27]:
if ORTOOLS_AVAILABLE:
    run_local_minimum_solver_self_tests()
else:
    msg = (
        "OR-Tools unavailable; cannot run local-minimum solver self-tests. "
        "Install with: %pip install ortools"
    )
    if HEADLESS or SMOKE:
        raise RuntimeError(msg)
    print(msg)


LocalMinimumSolver self-tests passed.


In [28]:
inst = generate_planted_span_instance(N=20, r=8, W=3, seed=0, hide=True)
bank = DirectionBank(inst.A)

solver = LocalMinimumSolver(num_search_workers=1, random_seed=0)
result = solver.solve(inst.A, inst.W, bank, time_limit_s=10)

print(result.status, result.found, result.weight)
verify_solver_result(inst.A, inst.W, bank, result)

OPTIMAL True 3


## 06. Failed-local-minima archive and attack utilities

In [ ]:

# @title Failed-local-minima archive and exact attack utilities
from dataclasses import field
from typing import List
import hashlib


def vector_hash(x) -> str:
    xb = as_binary_uint8(x, name="x", copy=False)
    if xb.ndim != 1:
        raise ValueError(f"x must be 1D, got {xb.shape}")
    return hashlib.sha256(np.ascontiguousarray(xb, dtype=np.uint8).tobytes()).hexdigest()


def bank_contains_direction(direction_bank, d) -> bool:
    db = as_binary_uint8(d, name="d", copy=False)
    if db.ndim != 1 or db.shape[0] != direction_bank.A.shape[0]:
        raise ValueError(f"d must have shape [{direction_bank.A.shape[0]}], got {db.shape}")
    key = np.ascontiguousarray(db, dtype=np.uint8).tobytes()
    return key in direction_bank.hashes


def descent_margin(c, d) -> int:
    cb = as_binary_uint8(c, name="c", copy=False)
    db = as_binary_uint8(d, name="d", copy=False)
    return int(2 * support_intersection(cb, db) - hamming_weight(db))


def is_descent_direction(c, d) -> bool:
    cb = as_binary_uint8(c, name="c", copy=False)
    db = as_binary_uint8(d, name="d", copy=False)
    return bool(directional_delta(cb, db) < 0 and hamming_weight(cb ^ db) != 0)


def local_minimum_report(c, direction_bank) -> dict:
    cb = as_binary_uint8(c, name="c", copy=False)
    direction_bank.verify()
    deltas = direction_bank.deltas(cb)
    active = []
    near_active = []
    num_valid_neg = 0
    for j in range(direction_bank.D.shape[0]):
        d = direction_bank.D[j]
        inter = support_intersection(cb, d)
        half = hamming_weight(d) // 2
        gap = half - inter
        if gap == 0:
            active.append(j)
        if gap in (0, 1):
            near_active.append(j)
        if deltas[j] < 0 and hamming_weight(cb ^ d) != 0:
            num_valid_neg += 1
    return {
        "weight": hamming_weight(cb),
        "num_directions": int(direction_bank.D.shape[0]),
        "min_delta": int(deltas.min()) if deltas.size else 0,
        "num_negative_directions": int(np.sum(deltas < 0)),
        "num_valid_negative_directions": int(num_valid_neg),
        "is_local_minimum": bool(num_valid_neg == 0),
        "active_constraints": active,
        "near_active_constraints": near_active,
    }


@dataclass
class FailedMinimumEntry:
    u_bad: np.ndarray
    c_bad: np.ndarray
    weight: int
    active_constraints: List[int]
    near_active_constraints: List[int]
    timestamp: int
    origin_action: str
    hash: str
    metadata: dict = field(default_factory=dict)


class FailedMinimaArchive:
    def __init__(self, A, W):
        A_bin = as_binary_uint8(A, name="A", copy=True)
        if A_bin.ndim != 2:
            raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
        N, r = A_bin.shape
        if N <= 0 or r <= 0:
            raise ValueError("A must have N>0 and r>0")
        W_int = int(W)
        if not (1 <= W_int <= N):
            raise ValueError("W must satisfy 1 <= W <= N")
        self.A = A_bin
        self.W = W_int
        self.entries = []
        self.hashes = set()
        self.timestamp_counter = 0

    def add(self, u_bad, c_bad, direction_bank, origin_action: str = "unknown", metadata: Optional[dict] = None) -> Optional[FailedMinimumEntry]:
        direction_bank.verify()
        if not np.array_equal(direction_bank.A, self.A):
            raise ValueError("direction_bank.A must match archive A")
        N, r = self.A.shape
        ub = as_binary_uint8(u_bad, name="u_bad", copy=True)
        cb = as_binary_uint8(c_bad, name="c_bad", copy=True)
        if ub.ndim != 1 or ub.shape != (r,):
            raise ValueError(f"u_bad must have shape ({r},), got {ub.shape}")
        if cb.ndim != 1 or cb.shape != (N,):
            raise ValueError(f"c_bad must have shape ({N},), got {cb.shape}")
        if not np.array_equal(gf2_matvec(self.A, ub), cb):
            raise ValueError("c_bad must equal gf2_matvec(A, u_bad)")
        wt = hamming_weight(cb)
        if wt == 0:
            raise ValueError("c_bad must be nonzero")
        if wt <= self.W:
            raise ValueError("c_bad must satisfy hamming_weight(c_bad) > W")
        report = local_minimum_report(cb, direction_bank)
        if not report["is_local_minimum"]:
            raise ValueError("state is not a failed local minimum under current bank")
        h = vector_hash(cb)
        if h in self.hashes:
            return None
        ent = FailedMinimumEntry(
            u_bad=ub,
            c_bad=cb,
            weight=wt,
            active_constraints=list(report["active_constraints"]),
            near_active_constraints=list(report["near_active_constraints"]),
            timestamp=int(self.timestamp_counter),
            origin_action=str(origin_action),
            hash=h,
            metadata={} if metadata is None else dict(metadata),
        )
        self.entries.append(ent)
        self.hashes.add(h)
        self.timestamp_counter += 1
        return ent

    def __len__(self) -> int:
        return len(self.entries)

    def get(self, idx: int) -> FailedMinimumEntry:
        return self.entries[idx]

    def latest(self) -> Optional[FailedMinimumEntry]:
        return None if len(self.entries) == 0 else self.entries[-1]

    def verify_entry(self, entry, direction_bank=None) -> None:
        N, r = self.A.shape
        ub = as_binary_uint8(entry.u_bad, name="entry.u_bad", copy=False)
        cb = as_binary_uint8(entry.c_bad, name="entry.c_bad", copy=False)
        if ub.ndim != 1 or ub.shape != (r,):
            raise ValueError(f"entry.u_bad must have shape ({r},), got {ub.shape}")
        if cb.ndim != 1 or cb.shape != (N,):
            raise ValueError(f"entry.c_bad must have shape ({N},), got {cb.shape}")
        if not np.array_equal(gf2_matvec(self.A, ub), cb):
            raise ValueError("entry must satisfy c_bad == gf2_matvec(A, u_bad)")
        if hamming_weight(cb) == 0:
            raise ValueError("entry.c_bad must be nonzero")
        if int(entry.weight) != hamming_weight(cb):
            raise ValueError("entry.weight must equal hamming_weight(c_bad)")
        if int(entry.weight) <= self.W:
            raise ValueError("entry.weight must be > W")
        if vector_hash(cb) != entry.hash:
            raise ValueError("entry hash mismatch")
        if direction_bank is not None:
            if not np.array_equal(direction_bank.A, self.A):
                raise ValueError("direction_bank.A must match archive A")
            report = local_minimum_report(cb, direction_bank)
            if not report["is_local_minimum"]:
                raise ValueError("entry is not a local minimum under provided bank")
            if list(entry.active_constraints) != list(report["active_constraints"]):
                raise ValueError("active constraints mismatch")
            if list(entry.near_active_constraints) != list(report["near_active_constraints"]):
                raise ValueError("near-active constraints mismatch")

    def verify(self, direction_bank=None) -> None:
        recomputed_hashes = set()
        for entry in self.entries:
            self.verify_entry(entry, direction_bank=direction_bank)
            h = vector_hash(entry.c_bad)
            if h in recomputed_hashes:
                raise ValueError("duplicate c_bad hash in entries")
            recomputed_hashes.add(h)
        if recomputed_hashes != self.hashes:
            raise ValueError("archive hash set mismatch")


def random_attack_failed_minimum(A, failed_entry, direction_bank, rng, num_trials: int = 1000) -> Optional[dict]:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f"A must be 2D, got {A_bin.shape}")
    if num_trials < 0:
        raise ValueError("num_trials must be nonnegative")
    c_bad = as_binary_uint8(failed_entry.c_bad, name="failed_entry.c_bad", copy=False)
    if c_bad.shape != (A_bin.shape[0],):
        raise ValueError("failed entry incompatible with A")
    if not np.array_equal(direction_bank.A, A_bin):
        raise ValueError("direction_bank.A must equal A")
    direction_bank.verify()

    r = A_bin.shape[1]
    for t in range(1, int(num_trials) + 1):
        v = rng.integers(0, 2, size=(r,), dtype=np.uint8)
        d = gf2_matvec(A_bin, v)
        if hamming_weight(d) == 0:
            continue
        if bank_contains_direction(direction_bank, d):
            continue
        if is_descent_direction(c_bad, d):
            delta = directional_delta(c_bad, d)
            new_weight = hamming_weight(c_bad ^ d)
            return {
                "v": v,
                "d": d,
                "delta": int(delta),
                "margin": int(descent_margin(c_bad, d)),
                "new_weight": int(new_weight),
                "old_weight": int(hamming_weight(c_bad)),
                "trials": int(t),
            }
    return None


def planted_attack_direction(inst, u_current) -> dict:
    inst.verify()
    u_curr = as_binary_uint8(u_current, name="u_current", copy=False)
    if u_curr.ndim != 1 or u_curr.shape != inst.u_star.shape:
        raise ValueError(f"u_current must have shape {inst.u_star.shape}, got {u_curr.shape}")
    c_current = inst.matvec(u_curr)
    v_star = u_curr ^ inst.u_star
    d_star = inst.matvec(v_star)
    c_target = as_binary_uint8(inst.c_star, name="inst.c_star", copy=False)
    if not np.array_equal(c_current ^ d_star, c_target):
        raise ValueError("exactness failed: c_current XOR d_star must equal c_star")
    delta = directional_delta(c_current, d_star)
    margin = descent_margin(c_current, d_star)
    return {
        "v": v_star,
        "d": d_star,
        "c_current": c_current,
        "c_target": c_target,
        "delta": int(delta),
        "margin": int(margin),
        "old_weight": int(hamming_weight(c_current)),
        "target_weight": int(hamming_weight(c_target)),
        "valid_descent": bool(is_descent_direction(c_current, d_star)),
    }


def run_failed_minima_archive_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)
    # 1) explicit failed local minimum
    A = np.eye(4, dtype=np.uint8)
    W = 1
    c_bad = np.array([1, 1, 0, 0], dtype=np.uint8)
    u_bad = c_bad.copy()
    bank = DirectionBank(A)
    bank.add_one(np.array([0, 0, 1, 0], dtype=np.uint8))
    bank.add_one(np.array([0, 0, 0, 1], dtype=np.uint8))
    archive = FailedMinimaArchive(A, W)
    e = archive.add(u_bad, c_bad, bank, origin_action="unit-test")
    assert e is not None
    assert len(archive) == 1
    assert archive.add(u_bad, c_bad, bank) is None
    archive.verify(bank)

    # 2) reject non-local state
    bad_bank = DirectionBank(A)
    bad_bank.add_one(np.array([1, 0, 0, 0], dtype=np.uint8))
    archive2 = FailedMinimaArchive(A, W)
    try:
        archive2.add(u_bad, c_bad, bad_bank)
        raise AssertionError("expected ValueError for non-local state")
    except ValueError:
        pass

    # 3) reject weight <= W
    archive3 = FailedMinimaArchive(A, 2)
    try:
        archive3.add(u_bad, c_bad, bank)
        raise AssertionError("expected ValueError for weight <= W")
    except ValueError:
        pass

    # 4) zero-target descent invalid
    c = np.array([1, 1, 0, 0], dtype=np.uint8)
    d = c.copy()
    assert directional_delta(c, d) < 0
    assert is_descent_direction(c, d) is False

    # 5) random attack utility
    prop = random_attack_failed_minimum(A, e, bank, rng, num_trials=200)
    assert prop is not None
    d_prop = prop["d"]
    assert hamming_weight(d_prop) > 0
    assert not bank_contains_direction(bank, d_prop)
    assert np.array_equal(gf2_matvec(A, prop["v"]), d_prop)
    assert directional_delta(c_bad, d_prop) < 0
    assert hamming_weight(c_bad ^ d_prop) > 0

    # 6) planted attack direction
    inst = generate_planted_span_instance(N=8, r=5, W=2, seed=int(rng.integers(10**9)), hide=True)
    for _ in range(128):
        u_current = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
        out = planted_attack_direction(inst, u_current)
        if not np.array_equal(out["c_current"], inst.c_star) and out["old_weight"] > inst.W:
            break
    assert np.array_equal(out["c_current"] ^ out["d"], inst.c_star)
    assert np.array_equal(gf2_matvec(inst.A, out["v"]), out["d"])
    assert out["delta"] == out["target_weight"] - out["old_weight"]
    if out["old_weight"] > inst.W and out["target_weight"] > 0:
        assert out["valid_descent"] is True

    # 7) local minimum report consistency
    report = local_minimum_report(c_bad, bank)
    for key in [
        "weight", "num_directions", "min_delta", "num_negative_directions",
        "num_valid_negative_directions", "is_local_minimum", "active_constraints", "near_active_constraints"
    ]:
        assert key in report
    deltas = [directional_delta(c_bad, bank.D[j]) for j in range(bank.D.shape[0])]
    assert report["num_negative_directions"] == int(sum(int(dlt < 0) for dlt in deltas))
    valid_neg = 0
    for j in range(bank.D.shape[0]):
        if directional_delta(c_bad, bank.D[j]) < 0 and hamming_weight(c_bad ^ bank.D[j]) != 0:
            valid_neg += 1
    assert report["num_valid_negative_directions"] == valid_neg

    # descent margin identity
    for j in range(bank.D.shape[0]):
        assert descent_margin(c_bad, bank.D[j]) == -directional_delta(c_bad, bank.D[j])

    print("run_failed_minima_archive_self_tests: PASS")


In [ ]:
run_failed_minima_archive_self_tests()

## 07. Neural direction ranker

In [ ]:
# @title Neural direction ranker (Torch scoring over exact direction features)
from typing import Optional

if TORCH_AVAILABLE:
    import torch.nn as nn

    class DirectionRanker(nn.Module):
        def __init__(self, dir_feat_dim, global_feat_dim, hidden_dim=128):
            super().__init__()
            self.dir_mlp = nn.Sequential(
                nn.Linear(dir_feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
            self.global_mlp = nn.Sequential(
                nn.Linear(global_feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
            self.score = nn.Sequential(
                nn.Linear(2 * hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1),
            )

        def forward(self, direction_features, global_features, mask=None):
            B, M, _ = direction_features.shape
            d_emb = self.dir_mlp(direction_features)
            g_emb = self.global_mlp(global_features).unsqueeze(1).expand(B, M, -1)
            scores = self.score(torch.cat([d_emb, g_emb], dim=-1)).squeeze(-1)
            if mask is not None:
                scores = scores.masked_fill(~mask.bool(), -1e9)
            return scores
else:
    class DirectionRanker:  # type: ignore[no-redef]
        def __init__(self, *args, **kwargs):
            raise RuntimeError("Torch unavailable; install torch or use Colab runtime with PyTorch.")


def global_features_for_ranker(A, W, c, bank) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    c_bin = as_binary_uint8(c, name="c", copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
    N, r = A_bin.shape
    if c_bin.ndim != 1 or c_bin.shape[0] != N:
        raise ValueError(f"c must have shape [{N}], got {c_bin.shape}")

    M = int(bank.D.shape[0])
    wt_c = hamming_weight(c_bin)
    deltas = bank.deltas(c_bin) if M > 0 else np.zeros((0,), dtype=np.int64)
    min_delta = int(np.min(deltas)) if deltas.size else 0
    num_negative = int(np.sum(deltas < 0)) if deltas.size else 0
    frac_negative = float(num_negative / max(1, M))

    feats = np.array([
        float(N),
        float(r),
        float(W),
        float(W) / max(1.0, float(N)),
        float(r) / max(1.0, float(N)),
        float(M),
        float(M) / max(1.0, float(N)),
        float(wt_c),
        float(wt_c) / max(1.0, float(W)),
        float(wt_c) / max(1.0, float(N)),
        float(min_delta),
        float(num_negative),
        frac_negative,
    ], dtype=np.float32)
    return feats


def rank_directions_with_model(model, c, bank, global_features, device="cpu", mask=None) -> np.ndarray:
    if bank.D.shape[0] == 0:
        return np.zeros((0,), dtype=np.int64)
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; install torch or use Colab runtime with PyTorch.")

    dir_features = bank.features_for_ranker(c).astype(np.float32, copy=False)
    global_features = np.asarray(global_features, dtype=np.float32)

    d_t = torch.from_numpy(dir_features).unsqueeze(0).to(device)
    g_t = torch.from_numpy(global_features).unsqueeze(0).to(device)
    m_t = None
    if mask is not None:
        m_t = torch.as_tensor(mask, device=device).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        scores = model(d_t, g_t, m_t).squeeze(0)
    ranking = torch.argsort(scores, descending=True)
    return ranking.detach().cpu().numpy().astype(np.int64)


def exact_best_direction_from_ranking(c, bank, ranking) -> Optional[int]:
    c_bin = as_binary_uint8(c, name="c", copy=False)
    for idx in np.asarray(ranking, dtype=np.int64):
        d = bank.D[int(idx)]
        if directional_delta(c_bin, d) < 0 and hamming_weight(c_bin ^ d) != 0:
            return int(idx)
    return None


In [ ]:
# @title Direction ranker self-tests (small deterministic smoke checks)
def make_direction_ranker_toy_batch(seed=0):
    rng = np.random.default_rng(seed)
    inst = generate_planted_span_instance(N=24, r=10, W=5, seed=seed, hide=True)
    bank = DirectionBank(inst.A)
    bank.add_random_span_directions(K=20, rng=rng)

    u = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
    while hamming_weight(u) == 0:
        u = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
    c = gf2_matvec(inst.A, u)

    dfeat = bank.features_for_ranker(c).astype(np.float32, copy=False)
    gfeat = global_features_for_ranker(inst.A, inst.W, c, bank)
    deltas = bank.deltas(c).astype(np.float32, copy=False)
    target = (-deltas / max(1.0, float(inst.A.shape[0]))).astype(np.float32)
    return inst, bank, c, dfeat, gfeat, target


def run_direction_ranker_self_tests(seed=0) -> None:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; install torch or use Colab runtime with PyTorch.")

    torch.manual_seed(seed)

    B, M, Fd, Fg = 2, 5, 6, 13
    model = DirectionRanker(Fd, Fg, hidden_dim=32)
    d = torch.randn(B, M, Fd)
    g = torch.randn(B, Fg)
    mask = torch.tensor([[1, 1, 0, 1, 0], [1, 0, 0, 1, 1]], dtype=torch.bool)
    scores = model(d, g, mask=mask)
    assert scores.shape == (B, M)
    assert torch.all(scores[~mask] <= -1e8)

    inst, bank, c, dfeat, gfeat, _ = make_direction_ranker_toy_batch(seed=seed)
    rk_model = DirectionRanker(dfeat.shape[1], gfeat.shape[0], hidden_dim=64)
    ranking = rank_directions_with_model(rk_model, c, bank, gfeat, device="cpu")
    assert ranking.shape == (bank.D.shape[0],)
    assert np.array_equal(np.sort(ranking), np.arange(bank.D.shape[0]))

    check_model = DirectionRanker(dfeat.shape[1], gfeat.shape[0], hidden_dim=32)
    forced = np.arange(bank.D.shape[0], dtype=np.int64)[::-1]
    idx = exact_best_direction_from_ranking(c, bank, forced)
    if idx is not None:
        assert directional_delta(c, bank.D[idx]) < 0
        assert hamming_weight(c ^ bank.D[idx]) != 0

    # Scores are only heuristic ranking signals; exact acceptance still requires delta<0 and nonzero next state.
    assert exact_best_direction_from_ranking(c, bank, ranking) is None or isinstance(
        exact_best_direction_from_ranking(c, bank, ranking), int
    )

    inst2, bank2, c2, dfeat2, gfeat2, target2 = make_direction_ranker_toy_batch(seed=seed + 1)
    train_model = DirectionRanker(dfeat2.shape[1], gfeat2.shape[0], hidden_dim=64)
    opt = torch.optim.Adam(train_model.parameters(), lr=1e-2)
    d_t = torch.from_numpy(dfeat2).unsqueeze(0)
    g_t = torch.from_numpy(gfeat2).unsqueeze(0)
    t_t = torch.from_numpy(target2).unsqueeze(0)

    train_model.train()
    with torch.no_grad():
        init_loss = torch.mean((train_model(d_t, g_t) - t_t) ** 2).item()
    for _ in range(30):
        opt.zero_grad()
        pred = train_model(d_t, g_t)
        loss = torch.mean((pred - t_t) ** 2)
        loss.backward()
        opt.step()
    final_loss = float(loss.item())
    assert final_loss < init_loss, (init_loss, final_loss)


if TORCH_AVAILABLE:
    run_direction_ranker_self_tests()
else:
    msg = "Torch unavailable; cannot run DirectionRanker self-tests."
    if HEADLESS or SMOKE:
        raise RuntimeError(msg)
    print(msg)


## 08. Neural coefficient generator

In [ ]:
# @title Neural coefficient generator (Torch logits over exact GF(2) coefficients)
from typing import List

if TORCH_AVAILABLE:
    import torch.nn as nn


if TORCH_AVAILABLE:
    class CoefficientGenerator(nn.Module):
        def __init__(self, basis_feat_dim, global_feat_dim, hidden_dim=128):
            super().__init__()
            self.basis_mlp = nn.Sequential(
                nn.Linear(basis_feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
            self.global_mlp = nn.Sequential(
                nn.Linear(global_feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
            self.out = nn.Sequential(
                nn.Linear(2 * hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1),
            )

        def forward(self, basis_features, global_features):
            B, r, _ = basis_features.shape
            b = self.basis_mlp(basis_features)
            g = self.global_mlp(global_features).unsqueeze(1).expand(B, r, -1)
            return self.out(torch.cat([b, g], dim=-1)).squeeze(-1)


def basis_features_for_generator(A, u_current, c_current) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    u_bin = as_binary_uint8(u_current, name="u_current", copy=False)
    c_bin = as_binary_uint8(c_current, name="c_current", copy=False)

    if A_bin.ndim != 2:
        raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
    N, r = A_bin.shape
    if u_bin.shape != (r,):
        raise ValueError(f"u_current must have shape ({r},), got {u_bin.shape}")
    if c_bin.shape != (N,):
        raise ValueError(f"c_current must have shape ({N},), got {c_bin.shape}")
    c_from_u = gf2_matvec(A_bin, u_bin)
    if not np.array_equal(c_from_u, c_bin):
        raise ValueError("c_current must equal gf2_matvec(A, u_current)")

    denom_n = max(1.0, float(N))
    denom_r = max(1.0, float(r - 1))
    feats = np.zeros((r, 5), dtype=np.float32)
    for j in range(r):
        col = A_bin[:, j]
        feats[j, 0] = hamming_weight(col) / denom_n
        feats[j, 1] = support_intersection(c_bin, col) / denom_n
        feats[j, 2] = directional_delta(c_bin, col) / denom_n
        feats[j, 3] = float(u_bin[j])
        feats[j, 4] = float(j) / denom_r
    return feats


def global_features_for_generator(A, W, u_current, c_current, bank=None) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    u_bin = as_binary_uint8(u_current, name="u_current", copy=False)
    c_bin = as_binary_uint8(c_current, name="c_current", copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
    N, r = A_bin.shape
    if u_bin.shape != (r,):
        raise ValueError(f"u_current must have shape ({r},), got {u_bin.shape}")
    if c_bin.shape != (N,):
        raise ValueError(f"c_current must have shape ({N},), got {c_bin.shape}")

    wc = hamming_weight(c_bin)
    vals = [
        float(N), float(r), float(W),
        float(W) / max(1.0, float(N)),
        float(r) / max(1.0, float(N)),
        float(wc),
        float(wc) / max(1.0, float(W)),
        float(wc) / max(1.0, float(N)),
    ]

    if bank is None:
        vals.extend([0.0, 0.0, 0.0, 0.0])
    else:
        M = int(bank.D.shape[0])
        if M == 0:
            vals.extend([0.0, 0.0, 0.0, 0.0])
        else:
            deltas = bank.deltas(c_bin)
            min_delta = float(np.min(deltas))
            neg_count = int(np.sum(deltas < 0))
            vals.extend([float(M), min_delta, float(neg_count), float(neg_count) / float(M)])
    return np.asarray(vals, dtype=np.float32)


def planted_coefficient_target(inst, u_current) -> np.ndarray:
    inst.verify()
    u_bin = as_binary_uint8(u_current, name="u_current", copy=False)
    r = inst.A.shape[1]
    if u_bin.shape != (r,):
        raise ValueError(f"u_current must have shape ({r},), got {u_bin.shape}")

    v_star = (u_bin ^ inst.u_star).astype(np.uint8, copy=False)
    lhs = gf2_matvec(inst.A, v_star)
    rhs = gf2_matvec(inst.A, u_bin) ^ inst.c_star
    if not np.array_equal(lhs, rhs):
        raise ValueError("Planted coefficient target identity failed")
    return v_star


def coefficient_direction_from_v(A, v, bank=None) -> dict:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    v_bin = as_binary_uint8(v, name="v", copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
    N, r = A_bin.shape
    if v_bin.shape != (r,):
        raise ValueError(f"v must have shape ({r},), got {v_bin.shape}")

    d = gf2_matvec(A_bin, v_bin)
    zero = hamming_weight(d) == 0
    duplicate = False if bank is None else bank_contains_direction(bank, d)
    return {
        "v": v_bin.copy(),
        "d": d,
        "direction_weight": hamming_weight(d),
        "is_zero_direction": bool(zero),
        "duplicate_in_bank": bool(duplicate),
    }


def sample_coefficients_from_logits(logits, num_samples=1, seed=None, temperature=1.0) -> np.ndarray:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; cannot sample from logits.")
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    t_logits = torch.as_tensor(logits, dtype=torch.float32)
    if t_logits.ndim not in (1, 2):
        raise ValueError(f"logits must have shape [r] or [B, r], got {tuple(t_logits.shape)}")
    gen = None
    if seed is not None:
        gen = torch.Generator(device="cpu")
        gen.manual_seed(int(seed))
    probs = torch.sigmoid(t_logits / float(temperature))
    if probs.ndim == 1:
        p = probs.unsqueeze(0).expand(int(num_samples), -1)
        out = torch.bernoulli(p, generator=gen)
        return out.to(dtype=torch.uint8).cpu().numpy()

    if int(num_samples) != 1:
        raise ValueError("For batched logits [B, r], only num_samples=1 is supported")
    out = torch.bernoulli(probs, generator=gen)
    return out.to(dtype=torch.uint8).cpu().numpy()


def propose_coefficients_with_model(
    model,
    A,
    W,
    u_current,
    c_current,
    bank=None,
    num_samples=1,
    device="cpu",
    seed=None,
    temperature=1.0,
) -> List[dict]:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; cannot propose coefficients with model.")
    bf = basis_features_for_generator(A, u_current, c_current)
    gf = global_features_for_generator(A, W, u_current, c_current, bank=bank)

    model = model.to(device)
    model.eval()
    with torch.no_grad():
        bf_t = torch.from_numpy(bf).unsqueeze(0).to(device=device, dtype=torch.float32)
        gf_t = torch.from_numpy(gf).unsqueeze(0).to(device=device, dtype=torch.float32)
        logits_t = model(bf_t, gf_t).squeeze(0).detach().cpu()

    v_samples = sample_coefficients_from_logits(
        logits_t,
        num_samples=num_samples,
        seed=seed,
        temperature=temperature,
    )

    probs = torch.sigmoid(logits_t / float(temperature)).cpu().numpy()
    proposals = []
    for v in v_samples:
        p = coefficient_direction_from_v(A, v, bank=bank)
        p["logits"] = logits_t.numpy().astype(np.float32, copy=True)
        p["probability_mean"] = float(np.mean(probs))
        # Heuristic proposal only; exact acceptance/validity checks happen elsewhere.
        proposals.append(p)
    return proposals


In [ ]:
# @title Coefficient generator self-tests (small deterministic smoke checks)
def run_coefficient_generator_self_tests(seed=0) -> None:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; install torch or use Colab runtime with PyTorch.")

    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    # 1) Shape test
    B, r, Fb, Fg = 3, 7, 5, 12
    model = CoefficientGenerator(Fb, Fg, hidden_dim=32)
    bf = torch.randn(B, r, Fb)
    gf = torch.randn(B, Fg)
    logits = model(bf, gf)
    assert logits.shape == (B, r)

    # 2) Planted target identity
    inst = generate_planted_span_instance(N=24, r=9, W=5, seed=seed, hide=True)
    for _ in range(6):
        u = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
        v_star = planted_coefficient_target(inst, u)
        lhs = gf2_matvec(inst.A, v_star)
        rhs = gf2_matvec(inst.A, u) ^ inst.c_star
        assert np.array_equal(lhs, rhs)

    # 3) Feature tests
    u0 = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
    c0 = gf2_matvec(inst.A, u0)
    bfeat = basis_features_for_generator(inst.A, u0, c0)
    gfeat = global_features_for_generator(inst.A, inst.W, u0, c0, bank=None)
    assert bfeat.shape[0] == inst.A.shape[1] and bfeat.dtype == np.float32
    assert gfeat.dtype == np.float32
    assert np.isfinite(bfeat).all() and np.isfinite(gfeat).all()

    # 4) Sampling tests
    test_logits = torch.tensor([0.1, -0.2, 0.0, 2.0], dtype=torch.float32)
    v_s = sample_coefficients_from_logits(test_logits, num_samples=5, seed=seed, temperature=1.0)
    assert v_s.shape == (5, 4)
    assert v_s.dtype == np.uint8 and np.isin(v_s, [0, 1]).all()

    inst2 = generate_planted_span_instance(N=12, r=4, W=3, seed=seed + 1, hide=True)
    d_info = coefficient_direction_from_v(inst2.A, v_s[0])
    assert np.array_equal(d_info["d"], gf2_matvec(inst2.A, v_s[0]))

    # 5) Proposal tests + no mutation
    bank = DirectionBank(inst.A)
    bank.add_random_span_directions(K=8, rng=rng)
    d_before = bank.D.copy()
    v_before = bank.V.copy()
    h_before = set(bank.hashes)
    model2 = CoefficientGenerator(bfeat.shape[1], gfeat.shape[0], hidden_dim=32)
    proposals = propose_coefficients_with_model(
        model2, inst.A, inst.W, u0, c0, bank=bank, num_samples=4, device="cpu", seed=seed
    )
    assert len(proposals) == 4
    for p in proposals:
        assert np.isin(p["v"], [0, 1]).all() and np.isin(p["d"], [0, 1]).all()
        assert np.array_equal(p["d"], gf2_matvec(inst.A, p["v"]))
        assert p["is_zero_direction"] == (hamming_weight(p["d"]) == 0)
        assert p["duplicate_in_bank"] == bank_contains_direction(bank, p["d"])
    assert np.array_equal(bank.D, d_before)
    assert np.array_equal(bank.V, v_before)
    assert bank.hashes == h_before

    # 6) Tiny supervised BCE loss-decrease test
    examples = []
    for i in range(2):
        inst_i = generate_planted_span_instance(N=20, r=8, W=4, seed=seed + 10 + i, hide=True)
        for _ in range(8):
            u_i = rng.integers(0, 2, size=(inst_i.A.shape[1],), dtype=np.uint8)
            c_i = gf2_matvec(inst_i.A, u_i)
            x_b = basis_features_for_generator(inst_i.A, u_i, c_i)
            x_g = global_features_for_generator(inst_i.A, inst_i.W, u_i, c_i, bank=None)
            y = planted_coefficient_target(inst_i, u_i).astype(np.float32)
            examples.append((x_b, x_g, y))

    Xb = torch.tensor(np.stack([e[0] for e in examples]), dtype=torch.float32)
    Xg = torch.tensor(np.stack([e[1] for e in examples]), dtype=torch.float32)
    Y = torch.tensor(np.stack([e[2] for e in examples]), dtype=torch.float32)

    train_model = CoefficientGenerator(Xb.shape[-1], Xg.shape[-1], hidden_dim=48)
    opt = torch.optim.Adam(train_model.parameters(), lr=1e-2)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    with torch.no_grad():
        initial_loss = float(loss_fn(train_model(Xb, Xg), Y).item())

    for _ in range(40):
        opt.zero_grad()
        loss = loss_fn(train_model(Xb, Xg), Y)
        loss.backward()
        opt.step()

    with torch.no_grad():
        final_loss = float(loss_fn(train_model(Xb, Xg), Y).item())
    assert final_loss < initial_loss, (initial_loss, final_loss)

    # 7) No validity claims: proposals are heuristic; d must be exact from A and v.
    for p in proposals:
        assert np.array_equal(p["d"], gf2_matvec(inst.A, p["v"]))


if TORCH_AVAILABLE:
    run_coefficient_generator_self_tests()
else:
    msg = "Torch unavailable; cannot run CoefficientGenerator self-tests."
    if HEADLESS or SMOKE:
        raise RuntimeError(msg)
    print(msg)


## 09. Cross-entropy sampler

In [ ]:

# @title Cross-entropy sampler (heuristic restart distribution over GF(2) coefficients)
from dataclasses import dataclass, field

@dataclass
class CrossEntropyCandidate:
    u_initial: np.ndarray
    u_final: np.ndarray
    c_final: np.ndarray
    weight: int
    descent_steps: int
    improved_by_descent: bool
    metadata: dict = field(default_factory=dict)


class CrossEntropySampler:
    def __init__(
        self,
        A,
        W: int,
        p_init=0.5,
        p_min: float = 0.01,
        p_max: float = 0.99,
        elite_frac: float = 0.2,
        smoothing: float = 0.5,
        seed: int = 0,
    ):
        A_bin = as_binary_uint8(A, name="A", copy=True)
        if A_bin.ndim != 2:
            raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
        N, r = A_bin.shape
        if N <= 0 or r <= 0:
            raise ValueError("A must have positive dimensions N and r")

        if not isinstance(W, (int, np.integer)):
            raise TypeError("W must be an integer")
        if not (1 <= int(W) <= N):
            raise ValueError(f"W must satisfy 1 <= W <= N, got W={W}, N={N}")

        if np.isscalar(p_init):
            p_arr = np.full((r,), float(p_init), dtype=np.float64)
        else:
            p_arr = np.asarray(p_init, dtype=np.float64)
            if p_arr.shape != (r,):
                raise ValueError(f"p_init array must have shape [{r}], got {p_arr.shape}")
        if np.any((p_arr < 0.0) | (p_arr > 1.0)):
            raise ValueError("p_init entries must be in [0, 1]")

        p_min = float(p_min)
        p_max = float(p_max)
        if not (0.0 <= p_min < p_max <= 1.0):
            raise ValueError("p_min and p_max must satisfy 0 <= p_min < p_max <= 1")

        elite_frac = float(elite_frac)
        if not (0.0 < elite_frac <= 1.0):
            raise ValueError("elite_frac must be in (0, 1]")

        smoothing = float(smoothing)
        if not (0.0 < smoothing <= 1.0):
            raise ValueError("smoothing must be in (0, 1]")

        self.A = A_bin
        self.N = N
        self.r = r
        self.W = int(W)
        self.p_min = p_min
        self.p_max = p_max
        self.elite_frac = elite_frac
        self.smoothing = smoothing
        self.p = np.clip(p_arr, self.p_min, self.p_max).astype(np.float64, copy=False)
        self.rng = np.random.default_rng(seed)
        self.best_seen: Optional[CrossEntropyCandidate] = None
        self.verify()

    def entropy(self) -> float:
        p = np.clip(self.p.astype(np.float64, copy=False), 1e-12, 1.0 - 1e-12)
        ent = -p * np.log(p) - (1.0 - p) * np.log(1.0 - p)
        return float(np.sum(ent, dtype=np.float64))

    def sample_coefficients(self, num_samples: int, rng=None) -> np.ndarray:
        if not isinstance(num_samples, (int, np.integer)) or int(num_samples) <= 0:
            raise ValueError("num_samples must be a positive integer")
        local_rng = self.rng if rng is None else rng
        U = local_rng.binomial(1, self.p.reshape(1, -1), size=(int(num_samples), self.r)).astype(np.uint8)
        return U

    def evaluate_candidate(self, u, bank=None, max_descent_steps: int = 0) -> CrossEntropyCandidate:
        u0 = as_binary_uint8(u, name="u", copy=True)
        if u0.ndim != 1 or u0.shape[0] != self.r:
            raise ValueError(f"u must have shape [{self.r}], got {u0.shape}")

        c = gf2_matvec(self.A, u0)
        u_work = u0.copy()
        steps = 0

        if bank is not None and int(max_descent_steps) > 0:
            bank.verify()
            if not np.array_equal(bank.A, self.A):
                raise ValueError("bank.A must equal sampler A")
            max_steps = int(max_descent_steps)
            while steps < max_steps:
                idx = bank.best_descent(c)
                if idx is None:
                    break
                d = bank.D[idx]
                v = bank.V[idx]
                c_new = c ^ d
                if hamming_weight(c_new) == 0:
                    break
                if hamming_weight(c_new) >= hamming_weight(c):
                    break
                u_work = u_work ^ v
                c = c_new
                steps += 1
                if not np.array_equal(c, gf2_matvec(self.A, u_work)):
                    raise AssertionError("coefficient-tracked invariant violated: c != A u")

        weight = hamming_weight(c)
        return CrossEntropyCandidate(
            u_initial=u0,
            u_final=u_work,
            c_final=c,
            weight=int(weight),
            descent_steps=int(steps),
            improved_by_descent=bool(steps > 0 and weight < hamming_weight(gf2_matvec(self.A, u0))),
            metadata={"heuristic": "cross_entropy_restart"},
        )

    def evaluate_batch(self, num_samples: int, bank=None, max_descent_steps: int = 0) -> list[CrossEntropyCandidate]:
        U = self.sample_coefficients(num_samples=num_samples)
        return [self.evaluate_candidate(U[i], bank=bank, max_descent_steps=max_descent_steps) for i in range(U.shape[0])]

    def select_elites(self, candidates, elite_frac=None, max_elites=None) -> list[CrossEntropyCandidate]:
        if len(candidates) == 0:
            raise ValueError("candidates must be nonempty")
        frac = self.elite_frac if elite_frac is None else float(elite_frac)
        if not (0.0 < frac <= 1.0):
            raise ValueError("elite_frac must be in (0, 1]")
        ordered = sorted(candidates, key=lambda c: (int(c.weight), -int(c.descent_steps)))
        k = max(1, int(np.ceil(frac * len(ordered))))
        if max_elites is not None:
            if int(max_elites) <= 0:
                raise ValueError("max_elites must be positive when provided")
            k = min(k, int(max_elites))
        return ordered[:k]

    def update_from_elites(self, elites, smoothing=None) -> dict:
        if len(elites) == 0:
            raise ValueError("elites must be nonempty")
        eta = self.smoothing if smoothing is None else float(smoothing)
        if not (0.0 < eta <= 1.0):
            raise ValueError("smoothing must be in (0, 1]")

        Ue = np.stack([as_binary_uint8(e.u_final, name="elite.u_final", copy=False) for e in elites], axis=0)
        if Ue.shape[1] != self.r:
            raise ValueError("elite coefficient dimension mismatch")
        elite_mean = Ue.mean(axis=0, dtype=np.float64)

        p_old = self.p.copy()
        entropy_old = self.entropy()
        p_new = (1.0 - eta) * p_old + eta * elite_mean
        p_new = np.clip(p_new, self.p_min, self.p_max)
        self.p = p_new.astype(np.float64, copy=False)
        entropy_new = self.entropy()
        self.verify()
        return {
            "p_old": p_old,
            "p_new": self.p.copy(),
            "elite_mean": elite_mean,
            "entropy_old": float(entropy_old),
            "entropy_new": float(entropy_new),
            "num_elites": int(len(elites)),
        }

    def best_seen_update(self, candidates) -> Optional[CrossEntropyCandidate]:
        if len(candidates) == 0:
            return self.best_seen
        best_batch = min(candidates, key=lambda c: (int(c.weight), -int(c.descent_steps)))
        if self.best_seen is None or (best_batch.weight, -best_batch.descent_steps) < (self.best_seen.weight, -self.best_seen.descent_steps):
            self.best_seen = best_batch
        return self.best_seen

    def verify(self) -> None:
        A = as_binary_uint8(self.A, name="A", copy=False)
        if A.ndim != 2:
            raise ValueError("A must be 2D")
        N, r = A.shape
        if N <= 0 or r <= 0:
            raise ValueError("A dimensions must be positive")
        if not isinstance(self.W, (int, np.integer)) or not (1 <= int(self.W) <= N):
            raise ValueError("W out of range")
        if self.p.shape != (r,):
            raise ValueError(f"p must have shape [{r}], got {self.p.shape}")
        if np.any(~np.isfinite(self.p)):
            raise ValueError("p contains non-finite values")
        if np.any((self.p < self.p_min) | (self.p > self.p_max)):
            raise ValueError("p out of configured bounds")
        if not (0.0 <= self.p_min < self.p_max <= 1.0):
            raise ValueError("invalid p_min/p_max")
        if not (0.0 < self.elite_frac <= 1.0):
            raise ValueError("invalid elite_frac")
        if not (0.0 < self.smoothing <= 1.0):
            raise ValueError("invalid smoothing")


def run_cross_entropy_round(
    sampler,
    num_samples,
    bank=None,
    max_descent_steps=0,
    elite_frac=None,
    smoothing=None,
) -> dict:
    candidates = sampler.evaluate_batch(num_samples=num_samples, bank=bank, max_descent_steps=max_descent_steps)
    elites = sampler.select_elites(candidates, elite_frac=elite_frac)
    update_info = sampler.update_from_elites(elites, smoothing=smoothing)
    best_candidate = sampler.best_seen_update(candidates)
    # Cross-entropy output is heuristic best-so-far only; no certification/optimality claim.
    return {
        "candidates": candidates,
        "elites": elites,
        "update_info": update_info,
        "best_candidate": best_candidate,
    }


In [ ]:

# @title Cross-entropy sampler self-tests (exact invariants + heuristic update checks)
def run_cross_entropy_sampler_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    # 1) Initialization and sampling.
    inst = generate_planted_span_instance(N=18, r=9, W=4, seed=seed, hide=True)
    sampler = CrossEntropySampler(inst.A, W=inst.W, p_init=0.5, seed=seed)
    sampler.verify()
    U = sampler.sample_coefficients(16)
    assert U.shape == (16, inst.A.shape[1])
    assert U.dtype == np.uint8
    assert np.all((U == 0) | (U == 1))

    # 2) Candidate evaluation exactness.
    cand = sampler.evaluate_candidate(U[0])
    assert np.array_equal(cand.c_final, gf2_matvec(inst.A, cand.u_final))
    assert cand.weight == hamming_weight(cand.c_final)

    # 3) Deterministic update formula + clipping.
    A_small = np.array([[1,0,1,0],[0,1,1,0],[1,1,0,1]], dtype=np.uint8)
    s2 = CrossEntropySampler(A_small, W=1, p_init=0.5, p_min=0.2, p_max=0.8, smoothing=0.5, seed=seed)
    elites = [
        CrossEntropyCandidate(np.zeros(4, dtype=np.uint8), np.array([1,0,1,0], dtype=np.uint8), np.zeros(3, dtype=np.uint8), 0, 0, False),
        CrossEntropyCandidate(np.zeros(4, dtype=np.uint8), np.array([1,1,1,0], dtype=np.uint8), np.zeros(3, dtype=np.uint8), 0, 0, False),
    ]
    p_old = s2.p.copy()
    info = s2.update_from_elites(elites, smoothing=0.5)
    elite_mean = np.array([1.0, 0.5, 1.0, 0.0], dtype=np.float64)
    expected = np.clip((1.0 - 0.5) * p_old + 0.5 * elite_mean, s2.p_min, s2.p_max)
    assert np.allclose(info["p_new"], expected, atol=1e-12)
    assert np.all((info["p_new"] >= s2.p_min) & (info["p_new"] <= s2.p_max))

    # 4) Entropy finite/nonnegative and changes after update.
    ent0 = float(info["entropy_old"])
    ent1 = float(info["entropy_new"])
    assert np.isfinite(ent0) and np.isfinite(ent1)
    assert ent0 >= 0.0 and ent1 >= 0.0
    assert abs(ent1 - ent0) > 1e-12

    # 5) Elite selection ordering and minimum 1.
    fake = [
        CrossEntropyCandidate(np.zeros(4, dtype=np.uint8), np.zeros(4, dtype=np.uint8), np.zeros(3, dtype=np.uint8), 7, 0, False),
        CrossEntropyCandidate(np.zeros(4, dtype=np.uint8), np.zeros(4, dtype=np.uint8), np.zeros(3, dtype=np.uint8), 3, 1, True),
        CrossEntropyCandidate(np.zeros(4, dtype=np.uint8), np.zeros(4, dtype=np.uint8), np.zeros(3, dtype=np.uint8), 3, 0, False),
    ]
    elites_small = sampler.select_elites(fake, elite_frac=0.01)
    assert len(elites_small) >= 1
    assert elites_small[0].weight == 3 and elites_small[0].descent_steps == 1

    # 6) Coefficient-tracked descent exactness.
    A_id = np.eye(4, dtype=np.uint8)
    bank = DirectionBank(A_id)
    bank.add_one(np.array([1,0,0,0], dtype=np.uint8))
    u_start = np.array([1,1,0,0], dtype=np.uint8)
    s3 = CrossEntropySampler(A_id, W=1, p_init=0.5, seed=seed)
    cand_d = s3.evaluate_candidate(u_start, bank=bank, max_descent_steps=4)
    assert cand_d.descent_steps >= 1
    assert cand_d.weight < hamming_weight(gf2_matvec(A_id, u_start))
    assert np.array_equal(cand_d.c_final, gf2_matvec(A_id, cand_d.u_final))

    # 7) No bank mutation.
    V_before = bank.V.copy()
    D_before = bank.D.copy()
    M_before = bank.D.shape[0]
    _ = s3.evaluate_candidate(u_start, bank=bank, max_descent_steps=2)
    _ = s3.evaluate_batch(5, bank=bank, max_descent_steps=2)
    assert bank.D.shape[0] == M_before
    assert np.array_equal(bank.V, V_before)
    assert np.array_equal(bank.D, D_before)

    # 8) Round helper contract and bounds.
    round_out = run_cross_entropy_round(sampler, num_samples=10, bank=None, max_descent_steps=0)
    for key in ["candidates", "elites", "update_info", "best_candidate"]:
        assert key in round_out
    assert np.all((sampler.p >= sampler.p_min) & (sampler.p <= sampler.p_max))
    bc = round_out["best_candidate"]
    assert bc is not None
    assert np.array_equal(bc.c_final, gf2_matvec(sampler.A, bc.u_final))

    # 9) No certification claim: heuristic best-so-far only.
    assert round_out["best_candidate"].metadata.get("heuristic") == "cross_entropy_restart"


run_cross_entropy_sampler_self_tests()


## 10. Strategy-level Q-controller skeleton

In [ ]:
# @title Strategy-level Q-controller skeleton (macro-action selector; exact algebra remains symbolic)
from dataclasses import dataclass, field
from typing import Optional

if TORCH_AVAILABLE:
    import torch
    import torch.nn as nn


if TORCH_AVAILABLE:
    class QControllerDQN(nn.Module):
        def __init__(self, obs_dim, num_actions=8, hidden_dim=256):
            super().__init__()
            self.trunk = nn.Sequential(
                nn.Linear(obs_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
            self.value = nn.Linear(hidden_dim, 1)
            self.advantage = nn.Linear(hidden_dim, num_actions)

        def forward(self, obs, action_mask=None):
            h = self.trunk(obs)
            v = self.value(h)
            a = self.advantage(h)
            q = v + a - a.mean(dim=-1, keepdim=True)
            if action_mask is not None:
                q = q.masked_fill(~action_mask.bool(), -1e9)
            return q


@dataclass
class SearchControllerState:
    inst: SpanInstance
    W: int
    u_current: np.ndarray
    c_current: np.ndarray
    bank: DirectionBank
    gradient_engine: GradientEngine
    archive: FailedMinimaArchive
    ce_sampler: CrossEntropySampler
    best_u: Optional[np.ndarray]
    best_c: Optional[np.ndarray]
    best_weight: int
    step: int
    terminal: bool = False
    terminal_reason: str = ""
    last_solver_result: Optional[SolverResult] = None
    last_action_info: dict = field(default_factory=dict)


@dataclass
class StepResult:
    observation: np.ndarray
    action_mask: np.ndarray
    reward: float
    done: bool
    info: dict


def _weight(x: np.ndarray) -> int:
    return int(np.sum(x, dtype=np.int64))


def initialize_controller_state(inst, M_init: int = 32, seed: int = 0, ce_sampler: Optional[CrossEntropySampler] = None) -> SearchControllerState:
    rng = np.random.default_rng(seed)
    bank = DirectionBank(inst.A)
    bank.add_random_span_directions(M_init, rng)
    bank.verify()
    gradient_engine = GradientEngine()
    archive = FailedMinimaArchive(inst.A, inst.W)
    archive.verify()
    if ce_sampler is None:
        ce_sampler = CrossEntropySampler(inst.A, W=inst.W, seed=seed)
    ce_sampler.verify()
    u0 = ce_sampler.sample_coefficients(1)[0]
    c0 = gf2_matvec(inst.A, u0)
    return SearchControllerState(inst=inst, W=inst.W, u_current=u0.copy(), c_current=c0.copy(), bank=bank, gradient_engine=gradient_engine, archive=archive, ce_sampler=ce_sampler, best_u=None, best_c=None, best_weight=inst.A.shape[0] + 1, step=0)


def controller_observation(state) -> np.ndarray:
    N, r = state.inst.A.shape
    c = state.c_current
    wcur = _weight(c)
    M = state.bank.D.shape[0]
    if M > 0:
        deltas = np.array([directional_delta(c, d) for d in state.bank.D], dtype=np.int64)
        dweights = np.array([_weight(d) for d in state.bank.D], dtype=np.float32)
        min_delta = float(np.min(deltas))
        nneg = int(np.sum(deltas < 0))
        frac_neg = float(nneg / max(1, M))
        dmin, dmean, dmax = float(np.min(dweights)), float(np.mean(dweights)), float(np.max(dweights))
    else:
        min_delta = 0.0
        nneg = 0
        frac_neg = 0.0
        dmin = dmean = dmax = 0.0
    best_w = state.best_weight if state.best_c is not None else 0
    failed_count = len(state.archive.entries)
    failed_best = min((e["weight"] for e in state.archive.entries), default=0)
    entropy = float(state.ce_sampler.entropy())
    solver_found = 1.0 if (state.last_solver_result is not None and state.last_solver_result.found) else 0.0
    solver_runtime = float(state.last_solver_result.runtime_s) if state.last_solver_result is not None else 0.0
    obs = np.array([
        N, r, state.W, state.W / max(1, N), r / max(1, N), M, M / max(1, N),
        wcur, wcur / max(1, state.W), wcur / max(1, N), best_w, best_w / max(1, state.W),
        min_delta, nneg, frac_neg, dmin, dmean, dmax, failed_count, failed_best, entropy,
        solver_found, solver_runtime, state.step, 1.0 if state.terminal else 0.0,
    ], dtype=np.float32)
    return obs


def controller_action_mask(state, direction_ranker=None, coefficient_generator=None, local_solver_available: Optional[bool] = None) -> np.ndarray:
    if local_solver_available is None:
        local_solver_available = ORTOOLS_AVAILABLE
    m = np.zeros(8, dtype=bool)
    m[0] = state.bank.best_descent(state.c_current) is not None
    m[1] = (direction_ranker is not None) and TORCH_AVAILABLE and (state.bank.D.shape[0] > 0)
    m[2] = True
    m[3] = (coefficient_generator is not None) and TORCH_AVAILABLE
    m[4] = bool(local_solver_available)
    m[5] = True
    m[6] = True
    m[7] = len(state.archive.entries) > 0
    return m


def choose_action_epsilon_greedy(q_model, observation, action_mask, epsilon: float = 0.1, rng=None, device="cpu") -> int:
    if rng is None:
        rng = np.random.default_rng(0)
    valid = np.where(action_mask)[0]
    if len(valid) == 0:
        raise RuntimeError("No available controller actions.")
    if rng.random() < epsilon:
        return int(rng.choice(valid))
    if q_model is None:
        return int(valid[0])
    obs_t = torch.tensor(observation[None, :], dtype=torch.float32, device=device)
    mask_t = torch.tensor(action_mask[None, :], dtype=torch.bool, device=device)
    with torch.no_grad():
        q = q_model(obs_t, action_mask=mask_t)[0].detach().cpu().numpy()
    return int(np.argmax(q))


def controller_step(state, action: int, rng=None, direction_ranker=None, coefficient_generator=None, K_add: int = 8, max_descent_steps: int = 5, solver_time_limit_s: float = 5.0) -> StepResult:
    if rng is None:
        rng = np.random.default_rng(0)
    mask = controller_action_mask(state, direction_ranker=direction_ranker, coefficient_generator=coefficient_generator)
    if not (0 <= int(action) <= 7):
        raise ValueError("Action must be in [0,7].")
    if not mask[action]:
        raise ValueError(f"Action {action} is currently masked out.")

    info = {"action": int(action), "moved": False}
    old_weight = _weight(state.c_current)
    old_best = state.best_weight

    if action == 0:
        idx = state.bank.best_descent(state.c_current)
        c_new, moved = state.gradient_engine.apply_best(state.c_current, state.bank)
        if moved and idx is not None:
            state.c_current = c_new
            state.u_current = (state.u_current ^ state.bank.V[idx]).astype(np.uint8)
            info.update({"moved": True, "idx": int(idx)})
        else:
            info["no_descent"] = True
    elif action == 1:
        N, r = state.inst.A.shape
        gfeat = np.array([N, r, state.W], dtype=np.float32)
        ranking = rank_directions_with_model(direction_ranker, state.c_current, state.bank, gfeat)
        idx = exact_best_direction_from_ranking(state.c_current, state.bank, ranking)
        if idx is not None:
            state.c_current = (state.c_current ^ state.bank.D[idx]).astype(np.uint8)
            state.u_current = (state.u_current ^ state.bank.V[idx]).astype(np.uint8)
            info.update({"moved": True, "idx": int(idx)})
        else:
            info["no_ranked_descent"] = True
    elif action == 2:
        prev = state.bank.D.shape[0]
        state.bank.add_random_span_directions(K_add, rng)
        info["added"] = state.bank.D.shape[0] - prev
    elif action == 3:
        N, r = state.inst.A.shape
        gfeat = np.array([N, r, state.W], dtype=np.float32)
        props = propose_coefficients_with_model(coefficient_generator, state.inst.A, global_features=gfeat, num_samples=K_add)
        added = 0
        for v in props:
            if state.bank.add_one(v):
                added += 1
        info["added"] = added
    elif action == 4:
        solver = LocalMinimumSolver(random_seed=int(rng.integers(1_000_000)))
        res = solver.solve(state.inst.A, state.W, state.bank, time_limit_s=solver_time_limit_s)
        state.last_solver_result = res
        info["solver_found"] = bool(res.found)
        if res.found:
            state.c_current = res.c.copy()
            state.u_current = res.u.copy()
            info["moved"] = True
    elif action == 5:
        u = state.ce_sampler.sample_coefficients(1)[0]
        state.u_current = u.copy()
        state.c_current = gf2_matvec(state.inst.A, state.u_current)
        info["restart"] = True
    elif action == 6:
        n_samples = 16 if (HEADLESS or SMOKE) else 32
        out = run_cross_entropy_round(state.ce_sampler, num_samples=n_samples, bank=state.bank, max_descent_steps=max_descent_steps)
        best = out["best_candidate"]
        info["ce_updated"] = True
        if best is not None and _weight(best.c_final) > 0 and _weight(best.c_final) < _weight(state.c_current):
            state.u_current = best.u_final.copy()
            state.c_current = best.c_final.copy()
            info["moved"] = True
    elif action == 7:
        target = state.archive.latest()
        if target is not None:
            proposal = random_attack_failed_minimum(state.inst.A, target["c"], state.bank, num_trials=16, rng=rng)
            if proposal is not None:
                added = state.bank.add_one(proposal["v"])
                info["added"] = int(bool(added))

    # Exact invariants
    assert np.array_equal(state.c_current, gf2_matvec(state.inst.A, state.u_current))
    state.bank.verify()

    w = _weight(state.c_current)
    if w > 0 and w < state.best_weight:
        state.best_weight = w
        state.best_c = state.c_current.copy()
        state.best_u = state.u_current.copy()

    report = None
    if w > 0 and w > state.W:
        report = local_minimum_report(state.c_current, state.bank)
        if report.get("is_local", False):
            state.archive.add(state.u_current, state.c_current, state.bank)

    if w > 0 and w <= state.W:
        state.terminal = True
        state.terminal_reason = "found_threshold_solution"

    reward = float(old_weight - w) + (10.0 if (w > 0 and w <= state.W) else 0.0) - 0.01
    state.step += 1
    state.last_action_info = info
    info.update({"old_weight": old_weight, "new_weight": w, "best_weight_before": old_best, "best_weight_after": state.best_weight, "terminal": state.terminal, "terminal_reason": state.terminal_reason})
    if report is not None:
        info["local_report"] = report
    return StepResult(observation=controller_observation(state), action_mask=controller_action_mask(state, direction_ranker, coefficient_generator), reward=reward, done=state.terminal, info=info)


def run_short_controller_episode(inst, num_steps: int = 8, seed: int = 0, use_models: bool = False) -> list[dict]:
    rng = np.random.default_rng(seed)
    state = initialize_controller_state(inst, seed=seed)
    trace = []
    for _ in range(num_steps):
        mask = controller_action_mask(state)
        a = int(rng.choice(np.where(mask)[0]))
        out = controller_step(state, a, rng=rng)
        row = {"action": a, "reward": out.reward, "current_weight": _weight(state.c_current), "best_weight": state.best_weight, "num_directions": int(state.bank.D.shape[0]), "failed_minima_count": len(state.archive.entries), "terminal": state.terminal, "terminal_reason": state.terminal_reason}
        trace.append(row)
        if state.terminal:
            break
    return trace


In [ ]:
# @title Q-controller self-tests (transition/coherence smoke only; no learning-quality claim)
def run_q_controller_self_tests(seed=0) -> None:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; cannot run Q-controller self-tests.")
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)

    Fo = 25
    model = QControllerDQN(obs_dim=Fo, num_actions=8, hidden_dim=64)
    obs = torch.randn(4, Fo)
    mask = torch.tensor([[1,1,1,1,1,1,1,1],[1,0,1,0,1,0,1,0],[0,1,0,1,0,1,0,1],[1,1,0,0,1,1,0,0]], dtype=torch.bool)
    q = model(obs, action_mask=mask)
    assert q.shape == (4, 8)
    assert torch.all(q[~mask] <= -1e8)

    inst = generate_planted_span_instance(N=24, r=10, W=4, seed=seed, hide=True)
    state = initialize_controller_state(inst, M_init=16, seed=seed)
    assert np.array_equal(state.c_current, gf2_matvec(inst.A, state.u_current))
    state.bank.verify(); state.archive.verify(); state.ce_sampler.verify()
    obs0 = controller_observation(state)
    assert np.all(np.isfinite(obs0))
    am = controller_action_mask(state)
    assert am.shape == (8,)

    assert am[2] and am[5] and am[6]
    assert am[4] == bool(ORTOOLS_AVAILABLE)
    assert not controller_action_mask(state, direction_ranker=None, coefficient_generator=None)[1]
    assert not controller_action_mask(state, direction_ranker=None, coefficient_generator=None)[3]

    A = np.eye(6, dtype=np.uint8)
    inst2 = SpanInstance(A=A, W=2, u_star=np.array([1,1,0,0,0,0],dtype=np.uint8), c_star=np.array([1,1,0,0,0,0],dtype=np.uint8), metadata={})
    s2 = initialize_controller_state(inst2, M_init=0, seed=seed)
    s2.u_current = np.array([1,1,1,0,0,0], dtype=np.uint8)
    s2.c_current = gf2_matvec(A, s2.u_current)
    s2.bank.add_one(np.array([0,0,1,0,0,0], dtype=np.uint8))
    w_old = _weight(s2.c_current)
    r0 = controller_step(s2, 0, rng=rng)
    assert np.array_equal(s2.c_current, gf2_matvec(A, s2.u_current))
    if r0.info.get("moved", False):
        assert _weight(s2.c_current) < w_old
        assert _weight(s2.c_current) > 0

    prev = state.bank.D.shape[0]
    controller_step(state, 2, rng=rng)
    assert state.bank.D.shape[0] >= prev
    state.bank.verify()

    controller_step(state, 5, rng=rng)
    assert np.array_equal(state.c_current, gf2_matvec(inst.A, state.u_current))
    controller_step(state, 6, rng=rng)
    assert np.all((state.ce_sampler.p >= 0.0) & (state.ce_sampler.p <= 1.0))
    assert np.array_equal(state.c_current, gf2_matvec(inst.A, state.u_current))

    if ORTOOLS_AVAILABLE:
        tiny = generate_planted_span_instance(N=12, r=6, W=3, seed=seed+1, hide=True)
        s3 = initialize_controller_state(tiny, M_init=4, seed=seed+1)
        o = controller_step(s3, 4, rng=rng, solver_time_limit_s=1.0)
        if o.info.get("solver_found", False):
            assert np.array_equal(s3.c_current, gf2_matvec(tiny.A, s3.u_current))
            assert _weight(s3.c_current) <= s3.W

    s4 = initialize_controller_state(inst2, M_init=0, seed=seed+3)
    s4.u_current = np.array([1,1,1,1,0,0], dtype=np.uint8)
    s4.c_current = gf2_matvec(A, s4.u_current)
    s4.bank.add_one(np.array([1,0,0,0,0,0], dtype=np.uint8))
    s4.bank.add_one(np.array([0,1,0,0,0,0], dtype=np.uint8))
    s4.bank.add_one(np.array([0,0,1,0,0,0], dtype=np.uint8))
    s4.bank.add_one(np.array([0,0,0,1,0,0], dtype=np.uint8))
    out4 = controller_step(s4, 2, rng=rng, K_add=0)
    if out4.info.get("local_report", {}).get("is_local", False):
        assert len(s4.archive.entries) >= 1

    tr = run_short_controller_episode(inst, num_steps=5, seed=seed)
    for row in tr:
        for k in ["action","reward","current_weight","best_weight","num_directions","failed_minima_count","terminal","terminal_reason"]:
            assert k in row
    print("Q-controller smoke trace:")
    for row in tr:
        print(row)
    print("Q-controller self-tests passed (transition/coherence smoke only; no learning-quality claim).")


if TORCH_AVAILABLE:
    run_q_controller_self_tests()
else:
    msg = "Torch unavailable; cannot run Q-controller self-tests."
    if HEADLESS or SMOKE:
        raise RuntimeError(msg)
    print(msg)


In [ ]:
# @title Run all notebook self-tests

def run_all_self_tests() -> None:
    run_f2_self_tests()
    run_planted_instance_self_tests()
    run_direction_bank_self_tests()
    run_gradient_engine_self_tests()
    if ORTOOLS_AVAILABLE:
        run_local_minimum_solver_self_tests()
    else:
        msg = "OR-Tools unavailable; cannot run local-minimum solver self-tests."
        if HEADLESS or SMOKE:
            raise RuntimeError(msg)
        print(msg)
    run_failed_minima_archive_self_tests()
    if TORCH_AVAILABLE:
        run_direction_ranker_self_tests()
        run_coefficient_generator_self_tests()
        run_q_controller_self_tests()
    else:
        msg = "Torch unavailable; cannot run neural/controller self-tests."
        if HEADLESS or SMOKE:
            raise RuntimeError(msg)
        print(msg)
    run_cross_entropy_sampler_self_tests()

run_all_self_tests()
